# Notebook 12 — Temporal Hyperparameter Search (LR / LightGBM)

在固定 NB10 的 Tree feature subset 与 NB11 的 LR representation 后，本 Notebook 对 LR 和 LightGBM 进行受约束的超参数搜索与 development 内部时间稳定性评估。

```text
NB10 Tree Feature Subset + NB11 LR Representation
                    ↓
Knowledge-guided Candidate Configurations
                    ↓
Recent-window Screening (Train 0–55 / Assess 56–63)
                    ↓
F1–F4 Temporal Robustness
                    ↓
AUC_MAX / ROBUST_1SE Selection Policies
                    ↓
Candidate Configurations for Downstream Validation
```

**数据边界：** 本册只使用 Week 0–63（development）。Week 64–77 Validation 和 Week 78–91 OOT 在此阶段均不用于训练、评估或选择。

**运行状态：** 保存的最后一步输出为 `Manual freeze disabled`，说明该次运行没有通过手动冻结单元导出两类 policy winner。候选评估及结果表仍保留，下游使用情况以具体 handoff 文件和后续 Notebook 记录为准。


## 1. 研究范围与执行流程

Notebook 12 不再回答“选哪些特征”或者“这些特征怎么表示”。

前面的工作已经把这两个问题拆开解决：

```text
NB10
Tree：选出固定 feature subset
LR：选出值得继续处理的 source space

        ↓

NB11
固定 LR representation
Raw Core vs Local FE vs PCA
→ 最终选择 Local FE

        ↓

NB12
只评估固定输入空间内的模型参数候选
```

因此 NB12 的任务可以拆成 6 个阶段：

```text
阶段 A：锁死时间边界
→ 只允许 Week 0–63

阶段 B：读取上游 Freeze
→ LR 读取 NB11 Local FE
→ Tree 读取 NB10 500 features

阶段 C：Knowledge-guided coarse search
→ LR 固定 L2 + LBFGS，搜索不同 C
→ Tree 少量复杂度 / 抽样 / 正则化配置

阶段 D：Recent-window 初筛
→ Train 0–55
→ Assess 56–63

阶段 E：Top candidates 做 F1–F4
→ 检查不同历史时间点能否重复

阶段 F：Policy selection / 可选人工 Freeze
→ 根据 F1–F4 汇总候选配置
→ 交给后续 Validation Notebook 比较
```

这里最重要的原则是：

> **NB12 不是重新寻找特征，而是在已经确定的特征空间上寻找一个稳定的模型配置。**


### 1.1 本册的时间数据角色

为了避免后面混淆，先把所有时间段的用途固定下来：

| 时间范围 | 在 NB12 的角色 | 是否可以用于调参 |
|---|---|---|
| Week 0–63 | Development | 可以 |
| Week 64–77 | Validation | **不可以，本册封存** |
| Week 78–91 | OOT | **不可以，本册封存** |

Development 内部再按时间顺序切：

```text
Stage 1:
0–55  →  56–63

F1:
0–31  →  32–39

F2:
0–39  →  40–47

F3:
0–47  →  48–55

F4:
0–55  →  56–63
```

箭头左边永远是“过去”，右边永远是“未来”。

这不是普通随机 Cross Validation，而是 **temporal validation**：模拟真实部署时“用过去预测未来”。


In [1]:
#@title 第 1 步：环境、开关与搜索预算
# ============================================================
# 1. Imports + constants + execution switches
# ============================================================

from pathlib import Path
from collections import Counter
import gc
import hashlib
import json
import math
import shutil
import time
import warnings

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import lightgbm as lgb

from scipy import sparse
from scipy.optimize import minimize
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, SplineTransformer
from IPython.display import display

ID_COL = "case_id"
TARGET_COL = "target"
WEEK_COL = "WEEK_NUM"
RANDOM_SEED = 42

DEVELOPMENT_WEEK_START = 0
DEVELOPMENT_WEEK_END = 63

STAGE1_TRAIN_START = 0
STAGE1_TRAIN_END = 55
STAGE1_ASSESS_START = 56
STAGE1_ASSESS_END = 63

TEMPORAL_FOLDS = [
    ("F1", 0, 31, 32, 39),
    ("F2", 0, 39, 40, 47),
    ("F3", 0, 47, 48, 55),
    ("F4", 0, 55, 56, 63),
]

# -------------------- execution switches --------------------
RUN_LR_COARSE = True
RUN_LR_FULL_TEMPORAL = True
RUN_TREE_COARSE = True
RUN_TREE_FULL_TEMPORAL = True

# 最终 Freeze 只冻结配置，不在 NB12 读取 Validation / OOT。
FREEZE_FINAL_MODELS = False
FINAL_LR_CANDIDATE = None
FINAL_TREE_CANDIDATE = None

# Dual-policy output:
# AUC_MAX = competition-oriented winner
# ROBUST_1SE = plateau -> calibration pass -> worst-fold AUC -> fold SD -> simplicity
# 两条 policy 共用同一批模型训练结果，不重复训练。


# -------------------- promotion budget --------------------
LR_PROMOTE_TOP_K = 3
TREE_PROMOTE_TOP_K = 5

# -------------------- calibration guardrails --------------------
# 这些是“宽松的项目级护栏”，不是普适统计定律。
# 它们只负责挡住明显失真的概率模型，不与 AUC 做加权总分。
CAL_SLOPE_MAX_ABS_DEV = 0.50        # slope roughly within [0.50, 1.50]
CAL_INTERCEPT_MAX_ABS = 0.50        # log-odds scale
OE_MAX_MULTIPLICATIVE_DEVIATION = 1.50  # O/E roughly within [1/1.5, 1.5]

# -------------------- runtime / IO --------------------
USE_LOCAL_TRAIN_CACHE = True
TREE_DEVICE_TYPE = "gpu"   # 如果 Colab 的 LightGBM GPU 不可用，改成 "cpu"
LGBM_MAX_BIN = 63
LGBM_BIN_SAMPLE_COUNT = 50_000
LGBM_MAX_CATEGORIES = 100
LGBM_ENABLE_BUNDLE = True

# Fast-LR 只缩小候选集合；评估器本身不变。
# 保持 evaluator version 不变，使之前已完成的相同 candidate checkpoint 可直接复用。
NB12_EVALUATOR_VERSION = "nb12_model_search_v3_dual_policy"


def assert_development_slice(start, end, label="development slice"):
    start, end = int(start), int(end)
    if start > end:
        raise ValueError(f"{label}: start > end")
    if start < DEVELOPMENT_WEEK_START or end > DEVELOPMENT_WEEK_END:
        raise ValueError(
            f"{label}: NB12 只能读取 Week {DEVELOPMENT_WEEK_START}–{DEVELOPMENT_WEEK_END}; "
            f"当前请求 {start}–{end}"
        )


def assert_temporal_split(tr0, tr1, va0, va1, label="split"):
    assert_development_slice(tr0, tr1, label + " train")
    assert_development_slice(va0, va1, label + " assess")
    if int(tr1) >= int(va0):
        raise ValueError(f"{label}: train 必须严格早于 assess")


assert_temporal_split(0, 55, 56, 63, "Stage 1")
for _fold in TEMPORAL_FOLDS:
    assert_temporal_split(*_fold[1:], _fold[0])

print("NB12 development-only temporal guardrail: OK")
print("Primary selection metric: AUC")
print("Calibration role: guardrail, not weighted objective")


NB12 development-only temporal guardrail: OK
Primary selection metric: AUC
Calibration role: guardrail, not weighted objective


## 2. Knowledge-guided Hyperparameter Candidate Space

这里不做大规模暴力 Grid Search，而是根据参数含义只测试少量有解释的候选。

### Logistic Regression

主要回答：

```text
C              → L2 正则化强弱（本次主要搜索维度）
penalty        → 固定 L2
solver         → 固定 L-BFGS
max_iter / tol → 收敛性诊断
```

由于 NB11 已出现 L-BFGS line-search warning，本册明确把 **convergence status** 作为筛选条件，而不是只比较 Gini。

### LightGBM

主要回答：

```text
num_leaves       → 树复杂度
min_data_in_leaf → 每个叶子至少多少样本
learning_rate    → 每棵树学多快
num_boost_round  → 总共多少棵树
feature_fraction → 每棵树随机看多少特征
bagging_fraction → 每棵树随机看多少样本
lambda_l1/l2     → 正则化
```


### LR 求解器及运行环境

NB12 的 LR 调参默认使用 **原生 scikit-learn `LogisticRegression`** 来获得明确的 `solver / max_iter / tol / ConvergenceWarning` 语义。

因此建议在一个**新的 Colab runtime**运行 NB12，不要先执行 `%load_ext cuml.accel`。NB11 的 cuML/L-BFGS line-search warning 本来就是本册要隔离并诊断的问题。Tree 分支仍然使用 LightGBM GPU。


## 3. 受约束的超参数搜索设计

### 3.0 Fast LR：固定 LBFGS + L2

LR 搜索采用 **Fast LR**：

```text
solver  = LBFGS
penalty = L2
C       = [0.03, 0.1, 0.3, 1, 3]
```

原因不是改变研究问题，而是缩小没有必要的计算成本。

当前 LR design 大约是：

```text
1.1M+ rows × 456 columns
```

在这个规模上，SAGA 明显比 LBFGS 慢很多；而前面已经观察到多个 LBFGS candidate 能正常收敛。

因此本册把 LR 的问题收敛成：

> 在已经冻结的 Local FE representation 上，什么样的 **L2 regularization strength C** 最适合 temporal generalization？

这样仍然保留：

```text
Stage 1 recent-window screen
→ Top 3
→ F1–F4
→ AUC_MAX
→ ROBUST_1SE
```

时间切分、calibration、dual-policy 逻辑都没有变化。

本次搜索采用 checkpoint 复用机制，以减少对同一候选的重复计算。


理论上可以做：

```text
很多 C
× 多个 penalty
× 多个 solver
× 很多 Tree leaves
× 很多 min_data_in_leaf
× 很多 learning_rate
× 很多 sampling
× 很多 regularization
```

但这样会迅速变成几百甚至几千个组合，而且其中很多组合没有明确实验价值。

本册采用 **Knowledge-guided search**：

> 先根据参数含义提出少量有方向的候选，再让数据验证这些方向。

### LR 候选的思路

`C` 控制正则化强弱：

```text
C 小
→ 正则更强
→ 系数更保守
→ 更可能稳定，但可能欠拟合

C 大
→ 正则更弱
→ 模型更自由
→ 可能提升 Gini，也更容易过拟合
```

`penalty`：

```text
L2
→ 本次固定使用的正则项，重点比较 C 的影响。
```

`solver`：

```text
lbfgs
→ 本次固定使用的求解器；保留并检查收敛诊断。
```

### Tree 候选的思路

不是每次同时把所有参数都改变，而是设计“有方向的实验”：

```text
baseline
→ 当前基准

leaves ↓ / min_data_in_leaf ↑
→ 测试更保守的树

leaves ↑
→ 测试更复杂的树

feature_fraction ↓
→ 测试随机抽特征

bagging_fraction ↓
→ 测试随机抽样本

lambda_l1 / lambda_l2 ↑
→ 测试更强正则

learning_rate ↓ + more rounds
→ 测试更慢、更细的 boosting
```

因此每一个候选都有一个可以解释的实验问题，而不是“机器随机试数字”。


### 3.1 本册最终不只输出一个 winner，而是两个预先定义的 Policy winner

模型只训练一次，但结果表会被两套规则读取：

```text
Policy A — AUC_MAX
→ 直接服务竞赛目标
→ mean F1–F4 AUC 最大

Policy B — ROBUST_1SE
→ 先进入 1-SE-style AUC plateau
→ plateau 内 calibration Pass/Fail
→ highest worst-fold AUC
→ lower fold AUC SD
→ simpler model
```

因此这不是：

```text
重新训练两套模型
```

而是：

```text
同一批训练结果
→ 两种 selection policy
```

额外运行成本几乎只剩下 DataFrame 筛选和排序。


In [2]:
#@title 第 2 步：定义 Knowledge-guided candidates
# ============================================================
# 2. Candidate configurations
# ============================================================

# LR Fast Mode：固定稳定、可解释的 L2 + LBFGS，只调正则化强度 C。
# 原因：
# - 当前设计矩阵约 114 万行 × 456 列；
# - SAGA 在该规模下明显过慢；
# - NB11 已经冻结 representation，NB12 只需高效搜索 regularization strength；
# - 这 5 个 C 覆盖 stronger -> weaker regularization。
#
# 注意：C=0.1 / 0.3 / 1.0 与上一版完全相同，
# evaluator version 也保持不变，因此已有 checkpoint 可以复用。
LR_CANDIDATES = [
    {"candidate": "lr_l2_lbfgs_c0p03", "solver": "lbfgs", "penalty": "l2", "C": 0.03, "max_iter": 1000, "tol": 1e-4},
    {"candidate": "lr_l2_lbfgs_c0p1",  "solver": "lbfgs", "penalty": "l2", "C": 0.1,  "max_iter": 1000, "tol": 1e-4},
    {"candidate": "lr_l2_lbfgs_c0p3",  "solver": "lbfgs", "penalty": "l2", "C": 0.3,  "max_iter": 1000, "tol": 1e-4},
    {"candidate": "lr_l2_lbfgs_c1",    "solver": "lbfgs", "penalty": "l2", "C": 1.0,  "max_iter": 1000, "tol": 1e-4},
    {"candidate": "lr_l2_lbfgs_c3",    "solver": "lbfgs", "penalty": "l2", "C": 3.0,  "max_iter": 1000, "tol": 1e-4},
]

# Tree: 不做所有参数的笛卡尔积，而是用少量“有方向”的配置。
TREE_BASE = {
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.0,
    "lambda_l2": 1.0,
    "num_boost_round": 300,
}


def tree_cfg(candidate, **updates):
    cfg = dict(TREE_BASE)
    cfg.update(updates)
    cfg["candidate"] = candidate
    return cfg


TREE_CANDIDATES = [
    tree_cfg("tree_baseline"),
    tree_cfg("tree_leaves15_leaf50", num_leaves=15, min_data_in_leaf=50),
    tree_cfg("tree_leaves31_leaf50", min_data_in_leaf=50),
    tree_cfg("tree_leaves31_leaf100", min_data_in_leaf=100),
    tree_cfg("tree_leaves63_leaf50", num_leaves=63, min_data_in_leaf=50),
    tree_cfg("tree_leaves63_leaf100", num_leaves=63, min_data_in_leaf=100),
    tree_cfg("tree_feature60", feature_fraction=0.6),
    tree_cfg("tree_bag60", bagging_fraction=0.6),
    tree_cfg("tree_subsample_both", feature_fraction=0.6, bagging_fraction=0.7),
    tree_cfg("tree_l2_5", lambda_l2=5.0),
    tree_cfg("tree_l1_1_l2_1", lambda_l1=1.0, lambda_l2=1.0),
    tree_cfg("tree_slow500", learning_rate=0.03, num_boost_round=500),
]

print("LR candidates:", len(LR_CANDIDATES))
print("Tree candidates:", len(TREE_CANDIDATES))
display(pd.DataFrame(LR_CANDIDATES))
display(pd.DataFrame(TREE_CANDIDATES))


LR candidates: 5
Tree candidates: 12


,candidate,solver,penalty,C,max_iter,tol
0,lr_l2_lbfgs_c0p03,lbfgs,l2,0.03,1000,0.0001
1,lr_l2_lbfgs_c0p1,lbfgs,l2,0.10,1000,0.0001
2,lr_l2_lbfgs_c0p3,lbfgs,l2,0.30,1000,0.0001
3,lr_l2_lbfgs_c1,lbfgs,l2,1.00,1000,0.0001
4,lr_l2_lbfgs_c3,lbfgs,l2,3.00,1000,0.0001


,learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,bagging_freq,lambda_l1,lambda_l2,num_boost_round,candidate
0,0.05,31,20,0.8,0.8,1,0.0,1.0,300,tree_baseline
1,0.05,15,50,0.8,0.8,1,0.0,1.0,300,tree_leaves15_leaf50
2,0.05,31,50,0.8,0.8,1,0.0,1.0,300,tree_leaves31_leaf50
3,0.05,31,100,0.8,0.8,1,0.0,1.0,300,tree_leaves31_leaf100
4,0.05,63,50,0.8,0.8,1,0.0,1.0,300,tree_leaves63_leaf50
5,0.05,63,100,0.8,0.8,1,0.0,1.0,300,tree_leaves63_leaf100
6,0.05,31,20,0.6,0.8,1,0.0,1.0,300,tree_feature60
7,0.05,31,20,0.8,0.6,1,0.0,1.0,300,tree_bag60
8,0.05,31,20,0.6,0.7,1,0.0,1.0,300,tree_subsample_both
9,0.05,31,20,0.8,0.8,1,0.0,5.0,300,tree_l2_5


## 4. 上游 Freeze：哪些东西已经不能再变？

运行 NB12 之前，要明确区分：

```text
已经冻结的东西
vs
NB12 仍然允许搜索的东西
```

### LR

NB11 会交给 NB12：

```text
selected_features_lr_raw_sources.json
selected_features_lr_engineered_specs.json
selected_features_lr_representation_spec.json
lr_nb12_handoff.json
```

也就是说：

```text
哪些 raw sources
+
增加哪些 Local FE
+
representation 是 Local FE
```

这些已经确定。

NB12 **不会再比较**：

```text
Raw vs Local FE vs PCA
top10 vs top20 vs top40
重新做 feature selection
```

它只比较：

```text
同一个 LR representation
+
不同模型超参数
```

### Tree

Tree 直接读取 NB10 Freeze 的：

```text
selected_features_tree.json
```

因此 Tree 调参时 feature subset 也保持固定。

这样做的目的，是把两个因素分开：

```text
Feature / Representation 的效果
≠
Model hyperparameter 的效果
```

否则如果一边换特征、一边换模型参数，就无法解释性能变化到底来自哪里。


In [3]:
#@title 第 3 步：挂载 Drive，定位 NB5 / NB7 / NB10 / NB11 输出
# ============================================================
# 3. Project paths + upstream handoff
# ============================================================

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError("Notebook 12 当前按 Google Colab + Google Drive 配置。") from exc

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/HomeCredit_Project")
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"

NB10_OUTPUT_DIR = ARTIFACT_ROOT / "nb10" / "final"
NB11_OUTPUT_DIR = ARTIFACT_ROOT / "nb11" / "final"
NB12_OUTPUT_DIR = ARTIFACT_ROOT / "nb12" / "final"
NB12_CHECKPOINT_DIR = ARTIFACT_ROOT / "nb12" / "checkpoints_v1"
NB12_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NB12_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def find_required_dir(required_files, preferred_candidates, label):
    required_files = set(required_files)
    for candidate in preferred_candidates:
        candidate = Path(candidate)
        if not candidate.exists() or not candidate.is_dir():
            continue
        try:
            names = {p.name for p in candidate.iterdir()}
        except OSError:
            continue
        if required_files.issubset(names):
            return candidate

    first_required = sorted(required_files)[0]
    for first_file in PROJECT_ROOT.rglob(first_required):
        parent = first_file.parent
        try:
            names = {p.name for p in parent.iterdir()}
        except OSError:
            continue
        if required_files.issubset(names):
            return parent

    raise FileNotFoundError(f"找不到 {label}: {sorted(required_files)}")


NB7_OUTPUT_DIR = find_required_dir(
    {"numeric_preprocessing_stats.csv", "categorical_encoding_rules.json", "nb7_candidate_features.csv"},
    [
        ARTIFACT_ROOT / "nb07" / "final",
        ARTIFACT_ROOT / "nb07",
        PROJECT_ROOT / "reports" / "reference_model",
    ],
    "Notebook 7 preprocessing",
)

NB5_OUTPUT_DIR = find_required_dir(
    {"train_clean.parquet", "feature_catalog.csv"},
    [
        ARTIFACT_ROOT / "nb05" / "final",
        ARTIFACT_ROOT / "nb05",
        PROJECT_ROOT / "clean_split",
    ],
    "Notebook 5 clean train",
)

# -------------------- LR handoff from NB11 --------------------
LR_HANDOFF_FILES = {
    "raw_sources": NB11_OUTPUT_DIR / "selected_features_lr_raw_sources.json",
    "engineered_specs": NB11_OUTPUT_DIR / "selected_features_lr_engineered_specs.json",
    "representation": NB11_OUTPUT_DIR / "selected_features_lr_representation_spec.json",
    "handoff": NB11_OUTPUT_DIR / "lr_nb12_handoff.json",
}

LR_READY = all(path.exists() for path in LR_HANDOFF_FILES.values())

if LR_READY:
    with open(LR_HANDOFF_FILES["raw_sources"], encoding="utf-8") as f:
        LR_RAW_SOURCES = json.load(f)
    with open(LR_HANDOFF_FILES["engineered_specs"], encoding="utf-8") as f:
        LR_ENGINEERED_SPECS = json.load(f)
    with open(LR_HANDOFF_FILES["representation"], encoding="utf-8") as f:
        LR_REPRESENTATION = json.load(f)
    with open(LR_HANDOFF_FILES["handoff"], encoding="utf-8") as f:
        LR_HANDOFF = json.load(f)

    if LR_HANDOFF.get("representation_mode") != "local_fe":
        raise RuntimeError(
            "NB12 当前按最终 Local FE 路线设计；"
            f"NB11 handoff 当前为 {LR_HANDOFF.get('representation_mode')}"
        )
    if not LR_HANDOFF.get("representation_frozen", False):
        raise RuntimeError("NB11 representation 尚未 freeze。")
else:
    LR_RAW_SOURCES = []
    LR_ENGINEERED_SPECS = []
    LR_REPRESENTATION = {}
    LR_HANDOFF = {}
    print("[LR SKIP] NB11 freeze 文件尚未齐全。等 NB11 freeze 完成后重新运行本 cell。")

# -------------------- Tree handoff from NB10 --------------------
TREE_FEATURE_PATH = NB10_OUTPUT_DIR / "selected_features_tree.json"
TREE_READY = TREE_FEATURE_PATH.exists()

if TREE_READY:
    with open(TREE_FEATURE_PATH, encoding="utf-8") as f:
        TREE_FEATURES = json.load(f)
else:
    TREE_FEATURES = []
    print(
        "[TREE SKIP] 找不到 selected_features_tree.json。\n"
        "请在 NB10 冻结 Tree recommendation：tree__relevance__n500，"
        "然后重新运行本 cell。"
    )

print("NB5:", NB5_OUTPUT_DIR)
print("NB7:", NB7_OUTPUT_DIR)
print("NB10:", NB10_OUTPUT_DIR)
print("NB11:", NB11_OUTPUT_DIR)
print("NB12:", NB12_OUTPUT_DIR)
print("LR_READY:", LR_READY, "| raw sources:", len(LR_RAW_SOURCES), "| engineered specs:", len(LR_ENGINEERED_SPECS))
print("TREE_READY:", TREE_READY, "| tree features:", len(TREE_FEATURES))


Mounted at /content/drive
NB5: /content/drive/MyDrive/HomeCredit_Project/artifacts/_auto_extracted/Notebook_5/nb05_clean_split_outputs/clean_split
NB7: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb07/final
NB10: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb10/final
NB11: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb11/final
NB12: /content/drive/MyDrive/HomeCredit_Project/artifacts/nb12/final
LR_READY: True | raw sources: 115 | engineered specs: 30
TREE_READY: True | tree features: 500


## 5. Temporal Validation 与信息隔离

这一部分是 NB12 最重要的方法学约束。

### 防线 1：代码层面锁死 Development Week 0–63

`assert_development_slice()` 会拒绝任何超出 Week 0–63 的请求。

因此 NB12 不能意外读取：

```text
Week 64–77 Validation
Week 78–91 OOT
```

### 防线 2：所有训练窗口必须严格早于评估窗口

`assert_temporal_split()` 强制：

```text
train_end < assess_start
```

因此不允许：

```text
未来 → 过去
```

也不允许 train / assess 时间重叠。

### 防线 3：Local FE 的“状态”只从当前 fold 的训练窗口学习

需要从数据学习参数的 FE，例如：

```text
winsor
→ clipping boundaries

quantile bins
→ bin edges

spline
→ knot / transformer state

ratio / log / product
→ 输出的 median / mean / scale
```

在每一个 fold 中都必须：

```text
只用 train window 拟合 state
        ↓
同一个 state transform train
        ↓
同一个 state transform assess
```

不能用 assess window 重新计算这些参数。

例如 F1：

```text
Week 0–31
→ 学 winsor boundary / qbin edges / spline state

Week 32–39
→ 只能应用 Week 0–31 学到的规则
```

### 防线 4：Tree categorical levels 也只从训练窗口学习

Tree 的类别变量也可能泄露。

例如未来出现了一个新的 category，如果先扫描完整 0–63 再建立 category mapping，就相当于提前知道未来有哪些类别。

所以每一个 fold：

```text
train window
→ 学 kept categories / rare categories

assess window
→ 只能应用 train window 的 category rules
```

未来没见过的类别统一映射到 `RARE`。

### 防线 5：不拿 assess window 做 early stopping

如果：

```text
0–55 训练
56–63 assessment
```

又用 56–63 来决定“训练到第几棵树停止”，那么 56–63 同时参与：

```text
模型训练决策
+
最终候选比较
```

这会让 assessment 不再干净。

所以本册没有用 assessment window 做 early stopping。

`num_boost_round` 直接作为候选超参数进行比较。

### 防线 6：最终 Validation / OOT 继续封存

NB12 的所有模型选择都完成以后，Week 64–77 和 78–91 仍然没有参与搜索。

这样 NB13 / 最终验收阶段仍然保留真正没有参与调参的数据。

> **时间泄露不只是“不要把未来行放进训练集”，还包括不要用未来数据学习预处理规则、类别规则、FE 参数或训练停止点。**


### 5.1 Calibration 指标在候选评估中的使用

不会，只要使用方式正确。

在 F1 中：

```text
Week 0–31
→ 训练模型

Week 32–39
→ 得到 prediction
→ 计算 AUC
→ 计算 calibration slope / intercept / O/E
```

Calibration 指标和 AUC 一样，都是在 **assessment window 上评估已经训练好的模型**。

真正会造成 leakage 的情况是：

```text
先看 assessment calibration
→ 再拿 assessment 数据重新拟合原模型参数
→ 然后还把同一个 assessment 当最终成绩
```

本册不这样做。

这里的 calibration 只用于：

```text
candidate evaluation / guardrail
```

而不会把 F1 的未来标签拿回去修改 F1 的训练矩阵或模型系数。

此外，Week 64–77 Validation 与 Week 78–91 OOT 仍然完全封存。


In [4]:
#@title 第 4 步：读取 NB7 预处理规则 + 本地缓存 Train parquet
# ============================================================
# 4. Data + frozen preprocessing
# ============================================================

numeric_stats = pd.read_csv(NB7_OUTPUT_DIR / "numeric_preprocessing_stats.csv")
nb7_catalog = pd.read_csv(NB7_OUTPUT_DIR / "nb7_candidate_features.csv")
with open(NB7_OUTPUT_DIR / "categorical_encoding_rules.json", encoding="utf-8") as f:
    lr_category_lists = json.load(f)

nb7_dtype = nb7_catalog.set_index("feature")["dtype_family"].to_dict()
numeric_stats_indexed = numeric_stats.set_index("feature")

DRIVE_TRAIN_PATH = NB5_OUTPUT_DIR / "train_clean.parquet"

if USE_LOCAL_TRAIN_CACHE:
    LOCAL_CACHE_ROOT = Path("/content/homecredit_nb12_cache")
    LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    LOCAL_TRAIN_PATH = LOCAL_CACHE_ROOT / DRIVE_TRAIN_PATH.name
    source_size = DRIVE_TRAIN_PATH.stat().st_size
    valid_local = LOCAL_TRAIN_PATH.exists() and LOCAL_TRAIN_PATH.stat().st_size == source_size
    if not valid_local:
        print("Caching train_clean.parquet to Colab local disk...")
        shutil.copy2(DRIVE_TRAIN_PATH, LOCAL_TRAIN_PATH)
    TRAIN_PATH = LOCAL_TRAIN_PATH
else:
    TRAIN_PATH = DRIVE_TRAIN_PATH

train_parquet = pq.ParquetFile(TRAIN_PATH)
train_meta = pd.read_parquet(
    TRAIN_PATH,
    columns=[ID_COL, WEEK_COL, TARGET_COL],
    engine="pyarrow",
).reset_index(drop=True)

if not train_meta[WEEK_COL].between(DEVELOPMENT_WEEK_START, DEVELOPMENT_WEEK_END).all():
    bad = train_meta.loc[~train_meta[WEEK_COL].between(0, 63), WEEK_COL].unique()[:20]
    raise RuntimeError(f"train_clean.parquet 出现 NB12 不允许的 week: {bad}")

ROW_GROUP_WEEK_CACHE = [
    train_parquet.read_row_group(i, columns=[WEEK_COL], use_threads=True)
    .column(0).to_numpy(zero_copy_only=False)
    for i in range(train_parquet.num_row_groups)
]

print("Train source:", TRAIN_PATH)
print("Rows:", len(train_meta), "| row groups:", train_parquet.num_row_groups)
print("Observed weeks:", int(train_meta[WEEK_COL].min()), "–", int(train_meta[WEEK_COL].max()))


Caching train_clean.parquet to Colab local disk...
Train source: /content/homecredit_nb12_cache/train_clean.parquet
Rows: 1282082 | row groups: 153
Observed weeks: 0 – 63


## 5. LR representation engine

这里**不重新搜索 Local FE**。

NB11 已经冻结：

```text
raw source list
+
engineered candidate specs
+
representation rule
```

NB12 对每个时间 fold 只做：

```text
读取 frozen specs
→ 在当前 fold 的 train window 拟合 FE state
→ transform train / assess
→ 得到固定 X
→ 多个 LR hyperparameter candidates 共用同一个 X
```

因此超参数比较不会重复进行 feature engineering，也不会让不同候选看到不同 representation。


### 6.1 Fold-specific Local FE Refit

NB11 Freeze 的不是一个“永远固定好的最终矩阵”。

它冻结的是：

```text
要使用哪些 raw sources
+
要做哪些 transformation
```

但某些 transformation 的数值参数仍然应该在当前训练窗口重新拟合。

所以每个 fold 的流程是：

```text
读取 frozen Local FE specs
        ↓
读取当前 train window
        ↓
fit_candidate_state(train)
        ↓
transform(train)
transform(assess)
        ↓
拼接：
Raw Core + Local FE
        ↓
得到这一折的 X_train / X_assess
```

例如一个 `qbin` 特征：

```text
F1:
Week 0–31 决定 quantile edges
Week 32–39 只使用这些 edges

F2:
Week 0–39 重新决定 quantile edges
Week 40–47 使用 F2 自己的 edges
```

这就是为什么：

> **representation 的“定义”被冻结，但 fold-specific fit state 不能从未来继承。**

同一个 fold 内，所有 LR 超参数候选共享完全相同的 `X_train / X_assess`，这样不同 LR candidate 的比较才公平。


In [5]:
#@title 第 5 步：LR frozen preprocessing + Local FE 重建函数
# ============================================================
# 5. LR design reconstruction
# ============================================================

LR_MISSING_TOKEN = "__HC_MISSING__"
LR_RARE_TOKEN = "__HC_RARE__"
LR_STATE_CACHE = {}
LR_SPLIT_CACHE = {}


def make_lr_state(features):
    """严格复用 NB7 已冻结的 raw LR preprocessing rule。"""
    cache_key = tuple(features)
    if cache_key in LR_STATE_CACHE:
        return LR_STATE_CACHE[cache_key]

    numeric = [f for f in features if nb7_dtype.get(f) == "numeric"]
    categorical = [f for f in features if nb7_dtype.get(f) in {"categorical", "boolean"}]

    stats = numeric_stats_indexed.reindex(numeric)
    required = ["median", "mean_after_imputation", "scale_after_imputation"]
    if numeric and stats[required].isna().any().any():
        missing = stats.index[stats[required].isna().any(axis=1)].tolist()
        raise ValueError(f"NB7 numerical preprocessing rule 缺失: {missing[:10]}")

    medians = stats["median"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)
    means = stats["mean_after_imputation"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)
    scales = stats["scale_after_imputation"].to_numpy(dtype=np.float32) if numeric else np.array([], dtype=np.float32)
    scales[~np.isfinite(scales) | (np.abs(scales) < 1e-12)] = 1.0

    missing_features = (
        stats.index[stats["has_missing"].fillna(False).astype(bool)].tolist()
        if numeric and "has_missing" in stats.columns else []
    )
    missing_indices = [numeric.index(f) for f in missing_features]

    category_lists = []
    kept_sets = {}
    for feature in categorical:
        if feature not in lr_category_lists:
            raise KeyError(f"NB7 categorical rule 缺少: {feature}")
        cats = [str(x) for x in lr_category_lists[feature]]
        if not cats:
            raise ValueError(f"NB7 categorical rule 为空: {feature}")
        category_lists.append(cats)
        kept_sets[feature] = set(cats)

    if categorical:
        kwargs = dict(categories=category_lists, handle_unknown="ignore", dtype=np.float32)
        try:
            encoder = OneHotEncoder(sparse_output=True, **kwargs)
        except TypeError:
            encoder = OneHotEncoder(sparse=True, **kwargs)
        dummy = pd.DataFrame({feature: [category_lists[i][0]] for i, feature in enumerate(categorical)})
        encoder.fit(dummy)
    else:
        encoder = None

    state = {
        "features": list(features),
        "numeric": numeric,
        "categorical": categorical,
        "medians": medians,
        "means": means,
        "scales": scales,
        "missing_indices": missing_indices,
        "kept_sets": kept_sets,
        "encoder": encoder,
    }
    LR_STATE_CACHE[cache_key] = state
    return state


def transform_lr_raw(frame, state):
    n = len(frame)
    numeric = state["numeric"]

    if numeric:
        X_num = frame[numeric].to_numpy(dtype=np.float32, na_value=np.nan)
        missing_mask = ~np.isfinite(X_num)
        if missing_mask.any():
            rows, cols = np.where(missing_mask)
            X_num[rows, cols] = state["medians"][cols]
        X_num -= state["means"]
        X_num /= state["scales"]

        if state["missing_indices"]:
            X_miss = sparse.csr_matrix(
                missing_mask[:, state["missing_indices"]].astype(np.float32),
                dtype=np.float32,
            )
        else:
            X_miss = sparse.csr_matrix((n, 0), dtype=np.float32)
    else:
        X_num = np.empty((n, 0), dtype=np.float32)
        X_miss = sparse.csr_matrix((n, 0), dtype=np.float32)

    categorical = state["categorical"]
    if categorical:
        mapped = {}
        for feature in categorical:
            s = frame[feature].astype("string").fillna(LR_MISSING_TOKEN)
            kept = state["kept_sets"][feature]
            s = s.where(s.isin(kept), LR_RARE_TOKEN)
            mapped[feature] = s.astype(object)
        X_cat = state["encoder"].transform(pd.DataFrame(mapped, index=frame.index))
    else:
        X_cat = sparse.csr_matrix((n, 0), dtype=np.float32)

    return sparse.hstack(
        [sparse.csr_matrix(X_num, dtype=np.float32), X_miss, X_cat],
        format="csr",
        dtype=np.float32,
    )


def _numeric_array(frame, feature):
    return pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)


def transform_candidate_raw(frame, spec, state):
    family = spec["family"]
    sources = spec["sources"]

    if family in {"signed_log1p", "winsor", "qbin", "spline"}:
        x = _numeric_array(frame, sources[0])

    if family == "signed_log1p":
        return np.sign(x) * np.log1p(np.abs(x))
    if family == "winsor":
        return np.clip(x, state["lower"], state["upper"])
    if family == "ratio":
        a = _numeric_array(frame, sources[0])
        b = _numeric_array(frame, sources[1])
        eps = float(spec.get("params", {}).get("epsilon", 1e-6))
        denom = np.where(np.abs(b) < eps, np.nan, b)
        return a / denom
    if family == "relative_change":
        a = _numeric_array(frame, sources[0])
        b = _numeric_array(frame, sources[1])
        eps = float(spec.get("params", {}).get("epsilon", 1e-6))
        denom = np.maximum(np.abs(b), eps)
        return (a - b) / denom
    if family == "product":
        a = _numeric_array(frame, sources[0])
        b = _numeric_array(frame, sources[1])
        return a * b
    raise ValueError(f"Unknown local FE family: {family}")


def fit_candidate_state(frame, spec):
    family = spec["family"]
    sources = list(spec["sources"])
    params = dict(spec.get("params", {}))
    state = {"family": family, "sources": sources, "params": params}

    if family == "winsor":
        x = _numeric_array(frame, sources[0])
        finite = x[np.isfinite(x)]
        state["lower"] = float(np.quantile(finite, params["lower_q"])) if len(finite) else 0.0
        state["upper"] = float(np.quantile(finite, params["upper_q"])) if len(finite) else 0.0

    elif family == "qbin":
        x = _numeric_array(frame, sources[0])
        finite = x[np.isfinite(x)]
        n_bins = int(params["n_bins"])
        if len(finite):
            qs = np.linspace(0, 1, n_bins + 1)[1:-1]
            edges = np.unique(np.quantile(finite, qs))
        else:
            edges = np.array([], dtype=float)
        state["edges"] = edges.tolist()

    elif family == "spline":
        x = _numeric_array(frame, sources[0])
        finite = x[np.isfinite(x)]
        median = float(np.median(finite)) if len(finite) else 0.0
        x2 = np.where(np.isfinite(x), x, median).reshape(-1, 1)
        transformer = SplineTransformer(
            n_knots=int(params["n_knots"]),
            degree=int(params["degree"]),
            knots="quantile",
            include_bias=False,
        )
        transformer.fit(x2)
        state["median"] = median
        state["transformer"] = transformer

    if family in {"signed_log1p", "winsor", "ratio", "relative_change", "product"}:
        raw = transform_candidate_raw(frame, spec, state)
        finite = raw[np.isfinite(raw)]
        median = float(np.median(finite)) if finite.size else 0.0
        raw2 = np.where(np.isfinite(raw), raw, median)
        mean = float(np.mean(raw2))
        scale = float(np.std(raw2))
        if not np.isfinite(scale) or scale < 1e-12:
            scale = 1.0
        state.update({"out_median": median, "out_mean": mean, "out_scale": scale})

    return state


def transform_candidate_matrix(frame, spec, state):
    family = spec["family"]

    if family == "qbin":
        x = _numeric_array(frame, spec["sources"][0])
        edges = np.asarray(state["edges"], dtype=float)
        bins = np.digitize(x, edges, right=False)
        bins[~np.isfinite(x)] = len(edges) + 1
        n_cols = len(edges) + 2
        rows = np.arange(len(frame))
        data = np.ones(len(frame), dtype=np.float32)
        return sparse.csr_matrix((data, (rows, bins)), shape=(len(frame), n_cols), dtype=np.float32)

    if family == "spline":
        x = _numeric_array(frame, spec["sources"][0])
        x2 = np.where(np.isfinite(x), x, state["median"]).reshape(-1, 1)
        out = state["transformer"].transform(x2).astype(np.float32)
        return sparse.csr_matrix(out)

    raw = transform_candidate_raw(frame, spec, state)
    raw = np.where(np.isfinite(raw), raw, state["out_median"])
    z = ((raw - state["out_mean"]) / state["out_scale"]).astype(np.float32)
    return sparse.csr_matrix(z.reshape(-1, 1))


def load_window_frame(features, week_start, week_end):
    """只读取指定 development window + 需要的 columns。"""
    assert_development_slice(week_start, week_end, "LR frame")
    features = list(dict.fromkeys(features))
    columns = [WEEK_COL, TARGET_COL, *features]
    parts = []

    for rg in range(train_parquet.num_row_groups):
        weeks = ROW_GROUP_WEEK_CACHE[rg]
        idx = np.flatnonzero((weeks >= week_start) & (weeks <= week_end)).astype(np.int32)
        if len(idx) == 0:
            continue
        table = train_parquet.read_row_group(rg, columns=columns, use_threads=True).take(pa.array(idx))
        parts.append(table.to_pandas())

    if not parts:
        raise RuntimeError(f"No rows for Week {week_start}–{week_end}")
    return pd.concat(parts, ignore_index=True)


def lr_required_sources():
    sources = set(LR_RAW_SOURCES)
    for spec in LR_ENGINEERED_SPECS:
        sources.update(spec["sources"])
    return sorted(sources)


def get_lr_split_design(train_start, train_end, assess_start, assess_end):
    """同一个 temporal split 只构建一次 Local FE design，后续 candidates 共用。"""
    if not LR_READY:
        raise RuntimeError("LR handoff not ready")

    key = (int(train_start), int(train_end), int(assess_start), int(assess_end))
    if key in LR_SPLIT_CACHE:
        return LR_SPLIT_CACHE[key]

    print(f"Building LR Local FE design: {train_start}-{train_end} -> {assess_start}-{assess_end}")
    required = lr_required_sources()
    train_df = load_window_frame(required, train_start, train_end)
    assess_df = load_window_frame(required, assess_start, assess_end)

    raw_state = make_lr_state(LR_RAW_SOURCES)
    X_train_blocks = [transform_lr_raw(train_df, raw_state)]
    X_assess_blocks = [transform_lr_raw(assess_df, raw_state)]

    for spec in LR_ENGINEERED_SPECS:
        state = fit_candidate_state(train_df, spec)
        X_train_blocks.append(transform_candidate_matrix(train_df, spec, state))
        X_assess_blocks.append(transform_candidate_matrix(assess_df, spec, state))

    X_train = sparse.hstack(X_train_blocks, format="csr", dtype=np.float32)
    X_assess = sparse.hstack(X_assess_blocks, format="csr", dtype=np.float32)
    y_train = train_df[TARGET_COL].to_numpy(dtype=np.int8)
    y_assess = assess_df[TARGET_COL].to_numpy(dtype=np.int8)
    assess_weeks = assess_df[WEEK_COL].to_numpy(dtype=np.int16)

    payload = (X_train, y_train, X_assess, y_assess, assess_weeks)
    LR_SPLIT_CACHE[key] = payload

    print("  design:", X_train.shape, "->", X_assess.shape)
    del train_df, assess_df, X_train_blocks, X_assess_blocks
    gc.collect()
    return payload


## 7. AUC 优先的模型选择目标

这个项目来自 Home Credit credit-risk competition，而竞赛核心评价方向是 **ranking / AUC**。

所以 NB12 不应该把：

```text
AUC
Calibration slope
Calibration intercept
O/E
```

做成四个“平起平坐”的加权目标。

这一版采用：

```text
AUC
→ 主目标

Temporal ranking stability
→ 稳定性检查

Calibration
→ probability guardrail

LR convergence
→ 有效性硬条件
```

### 7.1 Discrimination：AUC / Gini 在看什么？

AUC 回答：

> 模型能不能把真正更危险的客户排在更安全的客户前面？

这就是 **discrimination / ranking ability**。

Gini 与 AUC 完全单调等价：

```text
Gini = 2 × AUC - 1
```

所以两者不会产生不同的候选排序。

为了和 NB10 / NB11 保持连续，本册继续保存 Gini；  
但 **NB12 的 primary selection metric 改为 AUC**。

### 7.2 Temporal ranking stability 看什么？

总体 AUC 好，不代表每个未来时间窗口都好。

因此仍然记录：

```text
F1–F4 mean AUC
worst-fold AUC
fold AUC SD
weekly AUC slope
```

它们回答：

> 这个排序能力是不是只在某一个时间段碰巧很好？

### 7.3 Calibration 在这里不是“第二个主目标”

Calibration 回答的是：

> 预测出来的概率尺度是否合理？

理想状态：

```text
calibration slope     ≈ 1
calibration intercept ≈ 0
O/E                   ≈ 1
```

但本册不做：

```text
0.4 × AUC
+ 0.2 × slope
+ 0.2 × intercept
+ 0.2 × O/E
```

因为这种总分的权重非常主观，而且会把 ranking 和 probability accuracy 混成一个难解释的数字。

这里的原则是：

> **只要 calibration 没有差到明显失真，模型选择仍然以 AUC 为主。**

因此 calibration 是 **guardrail**，不是主优化目标。


In [6]:
#@title 第 6 步：统一指标、checkpoint 与 AUC-primary 稳健选择规则
# ============================================================
# 6. Metrics + calibration + checkpoint + recommendation
# ============================================================


def auc_from_pred(y, pred):
    if len(np.unique(y)) < 2:
        return np.nan
    return float(roc_auc_score(y, pred))


def gini_from_pred(y, pred):
    auc = auc_from_pred(y, pred)
    return float(2.0 * auc - 1.0) if np.isfinite(auc) else np.nan


def calibration_metrics(y, pred):
    """
    Assessment-window calibration diagnostics.

    calibration model:
        logit(P(Y=1)) = intercept + slope * logit(pred)

    Ideal:
        intercept ~= 0
        slope ~= 1
        O/E ~= 1

    注意：
    - 这是 assessment-window diagnostic；
    - 不用它重新训练原模型；
    - 不把这些指标和 AUC 加权成一个总分。
    """
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)

    if len(y) == 0 or len(np.unique(y)) < 2:
        return {
            "calibration_intercept": np.nan,
            "calibration_slope": np.nan,
            "observed_rate": np.nan,
            "expected_rate": np.nan,
            "oe_ratio": np.nan,
            "calibration_fit_success": False,
        }

    eps = 1e-6
    p = np.clip(pred, eps, 1.0 - eps)
    logit_p = np.log(p / (1.0 - p))

    def neg_log_likelihood(beta):
        eta = beta[0] + beta[1] * logit_p
        # stable Bernoulli negative log-likelihood
        return float(np.sum(np.logaddexp(0.0, eta) - y * eta))

    result = minimize(
        neg_log_likelihood,
        x0=np.array([0.0, 1.0], dtype=float),
        method="BFGS",
    )

    if not result.success or not np.all(np.isfinite(result.x)):
        # 一个更稳妥的 fallback
        result = minimize(
            neg_log_likelihood,
            x0=np.array([0.0, 1.0], dtype=float),
            method="L-BFGS-B",
        )

    if result.success and np.all(np.isfinite(result.x)):
        intercept = float(result.x[0])
        slope = float(result.x[1])
        fit_success = True
    else:
        intercept = np.nan
        slope = np.nan
        fit_success = False

    observed = float(np.mean(y))
    expected = float(np.mean(p))
    oe = float(observed / expected) if expected > 0 else np.nan

    return {
        "calibration_intercept": intercept,
        "calibration_slope": slope,
        "observed_rate": observed,
        "expected_rate": expected,
        "oe_ratio": oe,
        "calibration_fit_success": bool(fit_success),
    }


def calibration_guardrail(cal):
    """
    宽松护栏：只挡住明显失真的 probability calibration。
    阈值是项目级可编辑规则，不声称是行业统一标准。
    """
    slope = cal.get("calibration_slope", np.nan)
    intercept = cal.get("calibration_intercept", np.nan)
    oe = cal.get("oe_ratio", np.nan)

    finite = np.isfinite(slope) and np.isfinite(intercept) and np.isfinite(oe) and oe > 0
    if not finite:
        return False

    return bool(
        abs(float(slope) - 1.0) <= CAL_SLOPE_MAX_ABS_DEV
        and abs(float(intercept)) <= CAL_INTERCEPT_MAX_ABS
        and abs(math.log(float(oe))) <= math.log(OE_MAX_MULTIPLICATIVE_DEVIATION)
    )


def weekly_metrics(y, pred, weeks):
    rows = []
    for week in sorted(np.unique(weeks)):
        mask = weeks == week
        yw = y[mask]
        if len(np.unique(yw)) < 2:
            continue

        auc = auc_from_pred(yw, pred[mask])
        rows.append({
            "week": int(week),
            "auc": auc,
            "gini": float(2.0 * auc - 1.0),
            "rows": int(mask.sum()),
        })

    weekly = pd.DataFrame(rows)
    if weekly.empty:
        return weekly, {
            "mean_weekly_auc": np.nan,
            "weekly_auc_sd": np.nan,
            "weekly_auc_slope": np.nan,
            "mean_weekly_gini": np.nan,
            "weekly_gini_sd": np.nan,
            "weekly_gini_slope": np.nan,
        }

    x = weekly["week"].to_numpy(float)
    auc_y = weekly["auc"].to_numpy(float)
    gini_y = weekly["gini"].to_numpy(float)

    auc_slope = float(np.polyfit(x, auc_y, 1)[0]) if len(weekly) >= 2 else np.nan
    gini_slope = float(np.polyfit(x, gini_y, 1)[0]) if len(weekly) >= 2 else np.nan

    return weekly, {
        "mean_weekly_auc": float(weekly["auc"].mean()),
        "weekly_auc_sd": float(weekly["auc"].std(ddof=0)),
        "weekly_auc_slope": auc_slope,
        "mean_weekly_gini": float(weekly["gini"].mean()),
        "weekly_gini_sd": float(weekly["gini"].std(ddof=0)),
        "weekly_gini_slope": gini_slope,
    }


def fingerprint(payload):
    raw = json.dumps(payload, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:20]


def checkpoint_paths(branch, config, split):
    payload = {
        "version": NB12_EVALUATOR_VERSION,
        "branch": branch,
        "config": config,
        "split": list(map(int, split)),
        "lr_package": LR_HANDOFF.get("final_package") if LR_READY else None,
        "tree_feature_count": len(TREE_FEATURES) if TREE_READY else None,
        "calibration_guardrails": {
            "slope_max_abs_dev": CAL_SLOPE_MAX_ABS_DEV,
            "intercept_max_abs": CAL_INTERCEPT_MAX_ABS,
            "oe_max_multiplicative_deviation": OE_MAX_MULTIPLICATIVE_DEVIATION,
        },
    }
    key = fingerprint(payload)
    return (
        NB12_CHECKPOINT_DIR / f"{branch}__{key}__summary.json",
        NB12_CHECKPOINT_DIR / f"{branch}__{key}__weekly.csv",
    )



def _eligible_lr_rows(summary, branch):
    """
    LR: 如果存在 F1–F4 全部收敛候选，只在这些候选中选。
    Tree: 原样返回。
    """
    work = summary.copy()

    if branch == "lr" and "all_converged" in work.columns:
        converged = work[work["all_converged"] == True].copy()  # noqa: E712
        if not converged.empty:
            work = converged

    work = work[np.isfinite(work["mean_fold_auc"])].copy()
    return work


def auc_max_recommendation(summary, branch):
    """
    Policy A — AUC_MAX

    目标：
        直接服务竞赛 ranking metric。

    规则：
        1) LR 优先要求 F1–F4 全部收敛；
        2) mean F1–F4 AUC 最大；
        3) 如 mean AUC 完全相同，则 worst-fold AUC 更高；
        4) 再相同则 fold AUC SD 更小；
        5) 最后才用简单模型 tie-break。

    Calibration 在这条 policy 中只做 diagnostic，不阻止 winner。
    """
    if summary.empty:
        return None

    work = _eligible_lr_rows(summary, branch)
    if work.empty:
        return None

    sort_cols = [
        "mean_fold_auc",
        "worst_fold_auc",
        "fold_auc_sd",
    ]
    ascending = [False, False, True]

    if branch == "lr" and "C" in work.columns:
        sort_cols.append("C")
        ascending.append(True)

    if branch == "tree" and "num_leaves" in work.columns:
        sort_cols.append("num_leaves")
        ascending.append(True)

    chosen = work.sort_values(
        sort_cols,
        ascending=ascending,
    ).iloc[0]

    return {
        "policy": "AUC_MAX",
        "candidate": chosen["candidate"],
        "mean_fold_auc": float(chosen["mean_fold_auc"]),
        "worst_fold_auc": float(chosen["worst_fold_auc"]),
        "fold_auc_sd": float(chosen["fold_auc_sd"]),
        "mean_fold_gini": float(chosen["mean_fold_gini"]),
        "calibration_guardrail_pass": bool(chosen["calibration_guardrail_pass"]),
        "selection_rule": (
            "highest mean F1–F4 AUC -> highest worst-fold AUC -> "
            "lower fold AUC SD -> simpler tie-break"
        ),
        "note": (
            "Competition-oriented policy. Calibration is reported but does not "
            "override the AUC winner."
        ),
    }


def robust_1se_recommendation(summary, branch):
    """
    Policy B — ROBUST_1SE

    规则：
        1) LR 优先要求 F1–F4 全部收敛；
        2) 用 best candidate 的 fold AUC SD / sqrt(K)
           构造 1-SE-style AUC plateau；
        3) plateau 内不再追逐很小的 mean AUC 差异；
        4) calibration 只做 Pass / Fail；
        5) 通过者里选 highest worst-fold AUC；
        6) 再选 lower fold AUC SD；
        7) 再选 simpler model。

    重要：
        F1–F4 temporal folds 的 training windows 有重叠，
        所以这里只是 1-SE-style heuristic，
        不是严格独立 CV standard error。
    """
    if summary.empty:
        return None

    work = _eligible_lr_rows(summary, branch)
    if work.empty:
        return None

    best_row = (
        work
        .sort_values("mean_fold_auc", ascending=False)
        .iloc[0]
    )

    best_mean_auc = float(best_row["mean_fold_auc"])
    best_fold_auc_sd = float(best_row["fold_auc_sd"])
    k_folds = max(1, len(TEMPORAL_FOLDS))

    se_style_width = (
        best_fold_auc_sd / math.sqrt(k_folds)
        if np.isfinite(best_fold_auc_sd)
        else 0.0
    )

    plateau_threshold = (
        best_mean_auc
        - se_style_width
    )

    plateau = work[
        work["mean_fold_auc"]
        >= plateau_threshold
    ].copy()

    # Calibration 只作为 Pass/Fail gate。
    cal_pass = plateau[
        plateau["calibration_guardrail_pass"]
        == True  # noqa: E712
    ].copy()

    if not cal_pass.empty:
        survivors = cal_pass
        calibration_status = "PASS_FILTER_APPLIED"
        manual_review_required = False
    else:
        # 不偷偷改变规则：
        # 如果 plateau 内没人通过 calibration，
        # 暂时保留整个 plateau 供人工复核。
        survivors = plateau
        calibration_status = "NO_PASSING_CANDIDATE"
        manual_review_required = True

    # 一旦进入 plateau：
    # mean AUC 不再作为第一排序指标。
    sort_cols = [
        "worst_fold_auc",
        "fold_auc_sd",
    ]
    ascending = [False, True]

    if branch == "lr" and "C" in survivors.columns:
        sort_cols.append("C")
        ascending.append(True)

    if branch == "tree" and "num_leaves" in survivors.columns:
        sort_cols.append("num_leaves")
        ascending.append(True)

    # 只有在前面完全相同的情况下，mean AUC 才作为极末端 tie-break。
    sort_cols.append("mean_fold_auc")
    ascending.append(False)

    chosen = (
        survivors
        .sort_values(
            sort_cols,
            ascending=ascending,
        )
        .iloc[0]
    )

    return {
        "policy": "ROBUST_1SE",
        "candidate": chosen["candidate"],
        "mean_fold_auc": float(chosen["mean_fold_auc"]),
        "worst_fold_auc": float(chosen["worst_fold_auc"]),
        "fold_auc_sd": float(chosen["fold_auc_sd"]),
        "mean_fold_gini": float(chosen["mean_fold_gini"]),
        "best_mean_fold_auc": best_mean_auc,
        "best_fold_auc_sd": best_fold_auc_sd,
        "se_style_plateau_width": float(se_style_width),
        "auc_plateau_threshold": float(plateau_threshold),
        "plateau_candidate_count": int(len(plateau)),
        "calibration_guardrail_pass": bool(chosen["calibration_guardrail_pass"]),
        "calibration_status": calibration_status,
        "manual_review_required": bool(manual_review_required),
        "selection_rule": (
            "1-SE-style AUC plateau -> calibration Pass/Fail -> "
            "highest worst-fold AUC -> lower fold AUC SD -> "
            "simpler model -> mean AUC only as final tie-break"
        ),
        "important_note": (
            "The plateau width is heuristic because temporal folds overlap; "
            "this is not a strict independent-CV 1-SE rule."
        ),
    }


def build_dual_recommendations(summary, branch):
    return {
        "AUC_MAX": auc_max_recommendation(
            summary,
            branch,
        ),
        "ROBUST_1SE": robust_1se_recommendation(
            summary,
            branch,
        ),
    }


## 8. LR Candidate Evaluation Metrics

每个 LR candidate 现在会记录：

```text
1. Primary performance
   AUC
   Gini

2. Temporal ranking diagnostics
   weekly AUC
   weekly AUC SD
   weekly AUC slope

3. Calibration diagnostics
   calibration slope
   calibration intercept
   O/E

4. Optimization validity
   ConvergenceWarning
   n_iter
   converged
```

### 为什么 LR convergence 仍然是硬条件？

NB11 已经出现过：

```text
Maximum iterations reached
L-BFGS line search failed
```

因此一个看起来 AUC 很高但优化器没有可靠收敛的配置，不能直接当作最终 winner。

### 为什么 calibration 不直接决定 winner？

假设：

```text
Model A
AUC = 0.790
slope = 0.95
O/E = 1.04

Model B
AUC = 0.788
slope = 0.98
O/E = 1.01
```

两者 calibration 都没有明显问题。

本册不会因为 B 的 slope 更接近 1 就自动选 B。

仍然优先：

```text
AUC = 0.790
```

也就是 Model A。

但如果：

```text
Model C
AUC = 0.791
slope = 0.45
intercept = -0.70
O/E = 1.70
```

虽然 AUC 略高，probability calibration 已经明显异常。

这时 calibration guardrail 才真正发挥作用。


In [7]:
#@title 第 7 步：LR 单候选评估器（AUC + Calibration + Convergence）
# ============================================================
# 7. Logistic Regression evaluator
# ============================================================


def evaluate_lr_candidate(config, train_start, train_end, assess_start, assess_end):
    split = (train_start, train_end, assess_start, assess_end)
    summary_path, weekly_path = checkpoint_paths("lr", config, split)

    if summary_path.exists() and weekly_path.exists():
        with open(summary_path, encoding="utf-8") as f:
            return json.load(f), pd.read_csv(weekly_path)

    X_train, y_train, X_assess, y_assess, assess_weeks = get_lr_split_design(*split)

    model_kwargs = {
        "solver": config["solver"],
        "penalty": config["penalty"],
        "C": float(config["C"]),
        "max_iter": int(config["max_iter"]),
        "tol": float(config["tol"]),
        "random_state": RANDOM_SEED,
    }

    started = time.perf_counter()
    convergence_messages = []

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model = LogisticRegression(**model_kwargs)
        model.fit(X_train, y_train)

    for w in caught:
        if issubclass(w.category, ConvergenceWarning):
            convergence_messages.append(str(w.message))

    pred = model.predict_proba(X_assess)[:, 1]

    overall_auc = auc_from_pred(y_assess, pred)
    overall_gini = float(2.0 * overall_auc - 1.0) if np.isfinite(overall_auc) else np.nan

    calibration = calibration_metrics(y_assess, pred)
    calibration_pass = calibration_guardrail(calibration)

    weekly, weekly_summary = weekly_metrics(y_assess, pred, assess_weeks)

    n_iter = int(np.max(np.asarray(model.n_iter_))) if hasattr(model, "n_iter_") else None
    converged = (
        len(convergence_messages) == 0
        and (n_iter is None or n_iter < int(config["max_iter"]))
    )

    summary = {
        **config,
        "branch": "lr",
        "train_start": int(train_start),
        "train_end": int(train_end),
        "assess_start": int(assess_start),
        "assess_end": int(assess_end),
        "auc": float(overall_auc),
        "gini": float(overall_gini),
        **calibration,
        "calibration_guardrail_pass": bool(calibration_pass),
        **weekly_summary,
        "converged": bool(converged),
        "n_iter": n_iter,
        "convergence_warning_count": len(convergence_messages),
        "convergence_messages": convergence_messages[:5],
        "fit_seconds": float(time.perf_counter() - started),
        "design_columns": int(X_train.shape[1]),
    }

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    weekly.to_csv(weekly_path, index=False)
    return summary, weekly


## 9. LR Stage 1 — Recent-window Screening

如果 5 个 LR candidate 一开始全部跑 F1–F4：

```text
5 candidates × 4 folds
= 20 次完整实验
```

其中很多候选可能明显落后。

所以先用最近的 development window：

```text
Train 0–55
Assess 56–63
```

做一个便宜的 coarse screen。

它主要回答：

```text
哪些 C 候选
缺乏继续评估的价值？

哪些 candidate
有资格进入更贵的 F1–F4？
```

但这一阶段**不能直接决定最终模型**。

原因是：

> Week 56–63 只是一个时间窗口，某个 candidate 可能碰巧特别适合最近这段时间。

所以这里只做 promotion，不做 Freeze。


### Fast Mode 的 checkpoint 行为

如果上一版已经完成：

```text
lr_l2_lbfgs_c0p1
lr_l2_lbfgs_c0p3
lr_l2_lbfgs_c1
```

这三个候选，本版会直接读取相同 checkpoint，不重新 fit。

真正新增的 Stage 1 LR 训练只有：

```text
C = 0.03
C = 3.0
```

前提是 Google Drive 中原 checkpoint 文件仍然存在。


In [8]:
#@title 第 8 步：LR Stage 1 — recent-window coarse screen
# ============================================================
# 8. LR Fast Mode coarse screen: 0–55 -> 56–63
# ============================================================

lr_coarse = pd.DataFrame()

if RUN_LR_COARSE and LR_READY:
    rows = []

    # 同一 temporal split 的 Local FE design 只建一次，所有 LR 候选公平共用。
    get_lr_split_design(
        STAGE1_TRAIN_START,
        STAGE1_TRAIN_END,
        STAGE1_ASSESS_START,
        STAGE1_ASSESS_END,
    )

    for cfg in LR_CANDIDATES:
        print("LR Fast Stage 1:", cfg["candidate"])
        summary, _ = evaluate_lr_candidate(
            cfg,
            STAGE1_TRAIN_START,
            STAGE1_TRAIN_END,
            STAGE1_ASSESS_START,
            STAGE1_ASSESS_END,
        )
        rows.append(summary)

        print(
            f"  AUC={summary['auc']:.6f} | Gini={summary['gini']:.6f} "
            f"| cal_slope={summary['calibration_slope']:.3f} "
            f"| cal_intercept={summary['calibration_intercept']:.3f} "
            f"| O/E={summary['oe_ratio']:.3f} "
            f"| cal_guardrail={summary['calibration_guardrail_pass']} "
            f"| converged={summary['converged']} "
            f"| n_iter={summary['n_iter']} | {summary['fit_seconds']:.1f}s"
        )

    # Stage 1 promotion 仍以 AUC 为主；calibration 此处主要做诊断。
    lr_coarse = pd.DataFrame(rows).sort_values(
        ["converged", "auc", "weekly_auc_sd"],
        ascending=[False, False, True],
    ).reset_index(drop=True)

    lr_coarse.to_csv(NB12_OUTPUT_DIR / "lr_coarse_results.csv", index=False)
    display(lr_coarse)

elif (NB12_OUTPUT_DIR / "lr_coarse_results.csv").exists():
    lr_coarse = pd.read_csv(NB12_OUTPUT_DIR / "lr_coarse_results.csv")
    display(lr_coarse)

else:
    print("LR Stage 1 skipped.")


Building LR Local FE design: 0-55 -> 56-63
  design: (1148308, 456) -> (133774, 456)
LR Fast Stage 1: lr_l2_lbfgs_c0p03
  AUC=0.798884 | Gini=0.597767 | cal_slope=0.865 | cal_intercept=-0.308 | O/E=1.024 | cal_guardrail=True | converged=True | n_iter=86 | 72.2s
LR Fast Stage 1: lr_l2_lbfgs_c0p1
  AUC=0.798755 | Gini=0.597511 | cal_slope=0.860 | cal_intercept=-0.315 | O/E=1.028 | cal_guardrail=True | converged=True | n_iter=83 | 67.6s
LR Fast Stage 1: lr_l2_lbfgs_c0p3
  AUC=0.798850 | Gini=0.597699 | cal_slope=0.860 | cal_intercept=-0.320 | O/E=1.025 | cal_guardrail=True | converged=True | n_iter=115 | 96.2s
LR Fast Stage 1: lr_l2_lbfgs_c1
  AUC=0.798939 | Gini=0.597877 | cal_slope=0.862 | cal_intercept=-0.308 | O/E=1.032 | cal_guardrail=True | converged=True | n_iter=126 | 105.3s
LR Fast Stage 1: lr_l2_lbfgs_c3
  AUC=0.798941 | Gini=0.597883 | cal_slope=0.861 | cal_intercept=-0.323 | O/E=1.020 | cal_guardrail=True | converged=True | n_iter=108 | 89.7s


,candidate,solver,penalty,C,max_iter,tol,branch,train_start,train_end,assess_start,...,weekly_auc_slope,mean_weekly_gini,weekly_gini_sd,weekly_gini_slope,converged,n_iter,convergence_warning_count,convergence_messages,fit_seconds,design_columns
0,lr_l2_lbfgs_c3,lbfgs,l2,3.00,1000,0.0001,lr,0,55,56,...,-0.000623,0.598027,0.013797,-0.001247,True,108,0,[],89.742419,456
1,lr_l2_lbfgs_c1,lbfgs,l2,1.00,1000,0.0001,lr,0,55,56,...,-0.000620,0.598052,0.013856,-0.001240,True,126,0,[],105.322054,456
2,lr_l2_lbfgs_c0p03,lbfgs,l2,0.03,1000,0.0001,lr,0,55,56,...,-0.000468,0.597946,0.014120,-0.000937,True,86,0,[],72.191278,456
3,lr_l2_lbfgs_c0p3,lbfgs,l2,0.30,1000,0.0001,lr,0,55,56,...,-0.000624,0.597914,0.013976,-0.001248,True,115,0,[],96.211898,456
4,lr_l2_lbfgs_c0p1,lbfgs,l2,0.10,1000,0.0001,lr,0,55,56,...,-0.000508,0.597656,0.013957,-0.001016,True,83,0,[],67.612842,456


## 10. LR F1–F4 — Temporal Robustness

Stage 1 只晋级少量候选，例如 Top 3。

然后每个候选重新经历：

```text
F1  0–31 → 32–39
F2  0–39 → 40–47
F3  0–47 → 48–55
F4  0–55 → 56–63
```

每一折都会重新：

```text
构建该 fold 的 Local FE state
+
训练一个全新的 LR
+
在该 fold 的未来窗口预测
```

不会把 F4 训练出来的模型拿去评估 F1。

### 10.1 这一版的 1-SE-style AUC plateau

先找到：

```text
best mean F1–F4 AUC
```

再看这个最佳候选四个 fold 的 AUC 波动：

```text
fold_auc_sd
```

构造：

```text
SE-style width
=
fold_auc_sd / sqrt(4)
```

然后 near-best plateau 是：

```text
mean AUC
>=
best mean AUC - SE-style width
```

但是这里必须非常明确：

> **这不是严格统计意义上的独立-CV Standard Error。**

因为我们的 temporal folds：

```text
0–31
0–39
0–47
0–55
```

训练数据大量重叠，四折并不独立。

因此这里借用的是 **1-SE rule 的思想**：

> 不要为了非常小的 AUC 小数点差异，假装存在一个绝对确定的唯一最优参数。

Notebook 输出中会明确写成：

```text
1-SE-style / heuristic plateau
```

而不会声称这是严格的统计 SE。

### 10.2 Calibration 只在这个 AUC 平台里做护栏

进入 plateau 后检查：

```text
Calibration slope
Calibration intercept
O/E
```

如果存在 calibration 正常的候选：

```text
明显失真的候选
→ 从 plateau 排除
```

如果所有 near-best candidate 都 calibration 不理想：

```text
不强行假装找到“正常模型”
→ 保留 AUC-primary provisional recommendation
→ 标记 manual_review_required=True
→ 默认禁止自动 Freeze
```

### 10.3 通过校准护栏后的 Robustness 排序

`ROBUST_1SE` 首先限定在 near-best mean-AUC plateau 内，再优先比较：

```text
highest worst-fold AUC
↓
lower fold AUC SD
↓
simpler model
↓
mean AUC（最终 tie-break）
```

因此 **AUC_MAX** 与 **ROBUST_1SE** 的选择优先级不同：前者直接最大化 mean AUC，后者先满足近似 AUC 门槛，再比较最差时间表现与波动。校准指标作为筛选护栏而非加权目标。


In [9]:
#@title 第 9 步：LR Top candidates → F1–F4 full temporal
# ============================================================
# 9. LR full temporal validation
# ============================================================

lr_fold_results = pd.DataFrame()
lr_final_summary = pd.DataFrame()
lr_recommendations = {}

if RUN_LR_FULL_TEMPORAL and LR_READY and not lr_coarse.empty:
    eligible = lr_coarse[lr_coarse["converged"] == True].copy()  # noqa: E712

    if eligible.empty:
        print(
            "WARNING: Stage 1 没有完全收敛的 LR candidate；"
            "临时按 AUC promotion，但 Freeze 前必须人工复核。"
        )
        eligible = lr_coarse.copy()

    promoted_ids = (
        eligible
        .sort_values("auc", ascending=False)
        .head(LR_PROMOTE_TOP_K)["candidate"]
        .tolist()
    )

    cfg_lookup = {cfg["candidate"]: cfg for cfg in LR_CANDIDATES}
    print("LR Fast Mode promoted by Stage-1 AUC:", promoted_ids)

    rows = []

    for fold_name, tr0, tr1, va0, va1 in TEMPORAL_FOLDS:
        # 每个 fold 的 Local FE state 只从该 fold train window 学。
        get_lr_split_design(tr0, tr1, va0, va1)

        for candidate in promoted_ids:
            print("LR full temporal:", candidate, fold_name)
            summary, _ = evaluate_lr_candidate(
                cfg_lookup[candidate],
                tr0,
                tr1,
                va0,
                va1,
            )
            rows.append({"fold": fold_name, **summary})

    lr_fold_results = pd.DataFrame(rows)
    lr_fold_results.to_csv(
        NB12_OUTPUT_DIR / "lr_full_temporal_folds.csv",
        index=False,
    )

    aggregate_rows = []

    for candidate, group in lr_fold_results.groupby("candidate", sort=False):
        base = group.iloc[0]

        slope_dev = (group["calibration_slope"] - 1.0).abs()
        intercept_abs = group["calibration_intercept"].abs()

        oe_positive = group["oe_ratio"].where(group["oe_ratio"] > 0)
        abs_log_oe = np.log(oe_positive).abs()

        aggregate_rows.append({
            "candidate": candidate,
            "solver": base["solver"],
            "penalty": base["penalty"],
            "C": float(base["C"]),

            # Primary metric family
            "mean_fold_auc": float(group["auc"].mean()),
            "worst_fold_auc": float(group["auc"].min()),
            "fold_auc_sd": float(group["auc"].std(ddof=0)),

            # Gini retained for continuity with NB10/NB11
            "mean_fold_gini": float(group["gini"].mean()),
            "worst_fold_gini": float(group["gini"].min()),
            "fold_gini_sd": float(group["gini"].std(ddof=0)),

            # Temporal ranking stability
            "mean_weekly_auc": float(group["mean_weekly_auc"].mean()),
            "mean_abs_weekly_auc_slope": float(group["weekly_auc_slope"].abs().mean()),

            # Calibration diagnostics across F1–F4
            "mean_calibration_slope": float(group["calibration_slope"].mean()),
            "calibration_slope_sd": float(group["calibration_slope"].std(ddof=0)),
            "worst_calibration_slope_abs_dev": float(slope_dev.max()),

            "mean_calibration_intercept": float(group["calibration_intercept"].mean()),
            "calibration_intercept_sd": float(group["calibration_intercept"].std(ddof=0)),
            "worst_calibration_intercept_abs": float(intercept_abs.max()),

            "mean_oe_ratio": float(group["oe_ratio"].mean()),
            "oe_ratio_sd": float(group["oe_ratio"].std(ddof=0)),
            "worst_abs_log_oe": float(abs_log_oe.max()),

            "calibration_guardrail_pass": bool(group["calibration_guardrail_pass"].all()),
            "all_calibration_fits_successful": bool(group["calibration_fit_success"].all()),

            # Optimization validity
            "all_converged": bool(group["converged"].all()),
            "max_n_iter": int(group["n_iter"].max()),

            "total_fit_seconds": float(group["fit_seconds"].sum()),
            "design_columns": int(group["design_columns"].max()),
        })

    lr_final_summary = pd.DataFrame(aggregate_rows).sort_values(
        [
            "all_converged",
            "mean_fold_auc",
            "worst_fold_auc",
            "fold_auc_sd",
        ],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)

    lr_final_summary.to_csv(
        NB12_OUTPUT_DIR / "lr_final_temporal_summary.csv",
        index=False,
    )

    lr_recommendations = build_dual_recommendations(
        lr_final_summary,
        "lr",
    )

    display(lr_final_summary)

    print("LR AUC_MAX:", lr_recommendations["AUC_MAX"])
    print("LR ROBUST_1SE:", lr_recommendations["ROBUST_1SE"])

    with open(
        NB12_OUTPUT_DIR / "lr_dual_recommendations.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            lr_recommendations,
            f,
            ensure_ascii=False,
            indent=2,
        )

else:
    print("LR full temporal skipped.")


LR Fast Mode promoted by Stage-1 AUC: ['lr_l2_lbfgs_c3', 'lr_l2_lbfgs_c1', 'lr_l2_lbfgs_c0p03']
Building LR Local FE design: 0-31 -> 32-39
  design: (572238, 455) -> (182558, 455)
LR full temporal: lr_l2_lbfgs_c3 F1
LR full temporal: lr_l2_lbfgs_c1 F1
LR full temporal: lr_l2_lbfgs_c0p03 F1
Building LR Local FE design: 0-39 -> 40-47
  design: (754796, 455) -> (195725, 455)
LR full temporal: lr_l2_lbfgs_c3 F2
LR full temporal: lr_l2_lbfgs_c1 F2
LR full temporal: lr_l2_lbfgs_c0p03 F2
Building LR Local FE design: 0-47 -> 48-55
  design: (950521, 455) -> (197787, 455)
LR full temporal: lr_l2_lbfgs_c3 F3
LR full temporal: lr_l2_lbfgs_c1 F3
LR full temporal: lr_l2_lbfgs_c0p03 F3
LR full temporal: lr_l2_lbfgs_c3 F4
LR full temporal: lr_l2_lbfgs_c1 F4
LR full temporal: lr_l2_lbfgs_c0p03 F4


,candidate,solver,penalty,C,mean_fold_auc,worst_fold_auc,fold_auc_sd,mean_fold_gini,worst_fold_gini,fold_gini_sd,...,worst_calibration_intercept_abs,mean_oe_ratio,oe_ratio_sd,worst_abs_log_oe,calibration_guardrail_pass,all_calibration_fits_successful,all_converged,max_n_iter,total_fit_seconds,design_columns
0,lr_l2_lbfgs_c0p03,lbfgs,l2,0.03,0.789451,0.759682,0.017438,0.578902,0.519365,0.034877,...,1.677512,0.859415,0.318594,0.937712,False,True,True,92,205.843081,456
1,lr_l2_lbfgs_c3,lbfgs,l2,3.00,0.787463,0.750888,0.021178,0.574927,0.501776,0.042356,...,1.922901,0.871960,0.393109,1.109859,False,True,True,109,256.924056,456
2,lr_l2_lbfgs_c1,lbfgs,l2,1.00,0.787447,0.750547,0.021441,0.574895,0.501095,0.042881,...,1.930754,0.873679,0.379915,1.112587,False,True,True,126,250.891404,456


LR AUC_MAX: {'policy': 'AUC_MAX', 'candidate': 'lr_l2_lbfgs_c0p03', 'mean_fold_auc': 0.7894512457507447, 'worst_fold_auc': 0.7596823275089049, 'fold_auc_sd': 0.017438252151611103, 'mean_fold_gini': 0.5789024915014892, 'calibration_guardrail_pass': False, 'selection_rule': 'highest mean F1–F4 AUC -> highest worst-fold AUC -> lower fold AUC SD -> simpler tie-break', 'note': 'Competition-oriented policy. Calibration is reported but does not override the AUC winner.'}
LR ROBUST_1SE: {'policy': 'ROBUST_1SE', 'candidate': 'lr_l2_lbfgs_c0p03', 'mean_fold_auc': 0.7894512457507447, 'worst_fold_auc': 0.7596823275089049, 'fold_auc_sd': 0.017438252151611103, 'mean_fold_gini': 0.5789024915014892, 'best_mean_fold_auc': 0.7894512457507447, 'best_fold_auc_sd': 0.017438252151611103, 'se_style_plateau_width': 0.008719126075805552, 'auc_plateau_threshold': 0.780732119674939, 'plateau_candidate_count': 3, 'calibration_guardrail_pass': False, 'calibration_status': 'NO_PASSING_CANDIDATE', 'manual_review_req

## 10. Tree evaluator

Tree 分支继续使用 NB10 的思路：

- 固定 `selected_features_tree.json`；
- categorical level 只从当前 fold 的训练窗口学习；
- LightGBM 使用 memory-safe `Sequence` 读取 parquet；
- Stage 1 只比较 12 个知识引导候选；
- Top candidates 才进入 F1–F4。

本册**不使用 assessment window 做 early stopping**，避免把用于比较的未来窗口同时拿来决定训练停止点。`num_boost_round` 本身作为待比较的模型超参数之一。


## 11. Tree 与 LR 两条搜索路线

Tree 不需要 NB11 的 Local FE，因为 LightGBM 本身可以通过连续 split 学到很多非线性和 interaction。

所以 Tree 的路径更直接：

```text
NB10 frozen 500 features
        ↓
当前 fold train window
        ↓
学习 categorical rules
        ↓
LightGBM
        ↓
当前 fold future assessment
```

### Tree 这里主要在调什么？

可以分成四类：

**1. 模型复杂度**

```text
num_leaves
min_data_in_leaf
```

**2. Boosting 速度**

```text
learning_rate
num_boost_round
```

**3. 随机抽样**

```text
feature_fraction
bagging_fraction
```

**4. 正则化**

```text
lambda_l1
lambda_l2
```

这些参数虽然名字很多，但本质是在平衡：

```text
模型表达能力
vs
过拟合 / 时间不稳定
```


In [10]:
#@title 第 10 步：Tree streaming encoding / training / prediction
# ============================================================
# 10. Memory-safe LightGBM engine
# ============================================================

TREE_MISSING_TOKEN = "__MISSING__"
TREE_RARE_TOKEN = "__RARE__"
TREE_CATEGORY_RULE_CACHE = {}


def tree_feature_types():
    numeric = [f for f in TREE_FEATURES if nb7_dtype.get(f) == "numeric"]
    categorical = [f for f in TREE_FEATURES if nb7_dtype.get(f) in {"categorical", "boolean"}]
    unknown = [f for f in TREE_FEATURES if f not in set(numeric) | set(categorical)]
    if unknown:
        print("WARNING: dtype unknown for", len(unknown), "tree features; treating them as numeric.")
        numeric.extend(unknown)
    return numeric, categorical


def learn_tree_category_rules(week_start, week_end):
    key = (int(week_start), int(week_end))
    if key in TREE_CATEGORY_RULE_CACHE:
        return TREE_CATEGORY_RULE_CACHE[key]

    _, categorical_features = tree_feature_types()
    if not categorical_features:
        TREE_CATEGORY_RULE_CACHE[key] = {}
        return {}

    normal_limit = max(1, LGBM_MAX_CATEGORIES - 2)
    counters = {f: Counter() for f in categorical_features}

    batch_size = 10
    for start in range(0, len(categorical_features), batch_size):
        batch = categorical_features[start:start + batch_size]
        local_counters = {f: Counter() for f in batch}

        for rg in range(train_parquet.num_row_groups):
            weeks = ROW_GROUP_WEEK_CACHE[rg]
            idx = np.flatnonzero((weeks >= week_start) & (weeks <= week_end)).astype(np.int32)
            if len(idx) == 0:
                continue
            table = train_parquet.read_row_group(rg, columns=batch, use_threads=True).take(pa.array(idx))
            frame = table.to_pandas()
            for feature in batch:
                values = frame[feature].astype("string").fillna(TREE_MISSING_TOKEN)
                counts = values.value_counts()
                local_counters[feature].update({str(k): int(v) for k, v in counts.items()})
            del table, frame
            gc.collect()

        for feature in batch:
            counters[feature].update(local_counters[feature])

    rules = {}
    for feature in categorical_features:
        normal = [
            (cat, count) for cat, count in counters[feature].items()
            if cat not in {TREE_MISSING_TOKEN, TREE_RARE_TOKEN}
        ]
        normal.sort(key=lambda item: (-item[1], item[0]))
        kept = [cat for cat, _ in normal[:normal_limit]]
        categories = [*kept, TREE_MISSING_TOKEN, TREE_RARE_TOKEN]
        rules[feature] = {"categories": categories, "kept_set": set(categories)}

    TREE_CATEGORY_RULE_CACHE[key] = rules
    return rules


class TreeParquetSequence(lgb.Sequence):
    batch_size = 4096

    def __init__(self, week_start, week_end, category_rules):
        self.parquet_file = train_parquet
        self.numeric_features, self.categorical_features = tree_feature_types()
        self.feature_columns = [*self.numeric_features, *self.categorical_features]
        self.category_rules = category_rules
        self.selected_local_indices = []
        sizes = []

        for rg in range(self.parquet_file.num_row_groups):
            weeks = ROW_GROUP_WEEK_CACHE[rg]
            idx = np.flatnonzero((weeks >= week_start) & (weeks <= week_end)).astype(np.int32)
            self.selected_local_indices.append(idx)
            sizes.append(len(idx))

        self.selected_sizes = np.asarray(sizes, dtype=np.int64)
        self.selected_starts = np.concatenate([[0], np.cumsum(self.selected_sizes)]).astype(np.int64)
        self.total_rows = int(self.selected_starts[-1])
        self._cache_rg = None
        self._cache_matrix = None

    def __len__(self):
        return self.total_rows

    def _convert_row_group(self, rg):
        idx = self.selected_local_indices[rg]
        if len(idx) == 0:
            return np.empty((0, len(self.feature_columns)), dtype=np.float64)

        table = self.parquet_file.read_row_group(rg, columns=self.feature_columns, use_threads=True).take(pa.array(idx))
        frame = table.to_pandas()

        if self.numeric_features:
            numerical = frame[self.numeric_features].apply(pd.to_numeric, errors="coerce").to_numpy(
                dtype=np.float64, na_value=np.nan
            )
        else:
            numerical = np.empty((len(frame), 0), dtype=np.float64)

        cat_arrays = []
        for feature in self.categorical_features:
            values = frame[feature].astype("string").fillna(TREE_MISSING_TOKEN)
            rule = self.category_rules[feature]
            values = values.where(values.isin(rule["kept_set"]), TREE_RARE_TOKEN)
            codes = pd.Categorical(values, categories=rule["categories"]).codes.astype(np.float64)
            cat_arrays.append(codes)

        if cat_arrays:
            matrix = np.hstack([numerical, np.column_stack(cat_arrays).astype(np.float64, copy=False)])
        else:
            matrix = numerical

        del table, frame
        gc.collect()
        return np.asarray(matrix, dtype=np.float64, order="C")

    def _get_rg(self, rg):
        if self._cache_rg != rg:
            self._cache_matrix = self._convert_row_group(rg)
            self._cache_rg = rg
        return self._cache_matrix

    def _locate(self, selected_row_index):
        rg = int(np.searchsorted(self.selected_starts[1:], selected_row_index, side="right"))
        local = int(selected_row_index - self.selected_starts[rg])
        return rg, local

    def __getitem__(self, idx):
        if isinstance(idx, (int, np.integer)):
            if idx < 0:
                idx += len(self)
            rg, local = self._locate(int(idx))
            return self._get_rg(rg)[local]

        if isinstance(idx, slice):
            start = 0 if idx.start is None else int(idx.start)
            stop = len(self) if idx.stop is None else int(idx.stop)
            parts = []
            current = start
            while current < stop:
                rg, local = self._locate(current)
                matrix = self._get_rg(rg)
                take = min(len(matrix) - local, stop - current)
                parts.append(matrix[local:local + take])
                current += take
            if not parts:
                return np.empty((0, len(self.feature_columns)), dtype=np.float64)
            return parts[0] if len(parts) == 1 else np.vstack(parts)

        if isinstance(idx, (list, np.ndarray)):
            return np.stack([self[int(i)] for i in idx])

        raise TypeError(f"Unsupported index type: {type(idx)}")


def fit_tree_booster(config, train_start, train_end):
    rules = learn_tree_category_rules(train_start, train_end)
    seq = TreeParquetSequence(train_start, train_end, rules)
    mask = train_meta[WEEK_COL].between(train_start, train_end)
    y = train_meta.loc[mask, TARGET_COL].to_numpy(dtype=np.int8)
    assert len(seq) == len(y)

    categorical_indices = list(range(len(seq.numeric_features), len(seq.feature_columns)))
    dataset = lgb.Dataset(
        seq,
        label=y,
        feature_name=seq.feature_columns,
        categorical_feature=categorical_indices,
        free_raw_data=True,
        params={
            "bin_construct_sample_cnt": LGBM_BIN_SAMPLE_COUNT,
            "max_bin": LGBM_MAX_BIN,
            "enable_bundle": LGBM_ENABLE_BUNDLE,
            "data_random_seed": RANDOM_SEED,
        },
    )

    params = {
        "objective": "binary",
        "metric": "auc",
        "learning_rate": float(config["learning_rate"]),
        "num_leaves": int(config["num_leaves"]),
        "min_data_in_leaf": int(config["min_data_in_leaf"]),
        "feature_fraction": float(config["feature_fraction"]),
        "bagging_fraction": float(config["bagging_fraction"]),
        "bagging_freq": int(config["bagging_freq"]),
        "lambda_l1": float(config["lambda_l1"]),
        "lambda_l2": float(config["lambda_l2"]),
        "seed": RANDOM_SEED,
        "feature_fraction_seed": RANDOM_SEED,
        "bagging_seed": RANDOM_SEED,
        "verbosity": -1,
        "num_threads": -1,
        "device_type": TREE_DEVICE_TYPE,
        "max_bin": LGBM_MAX_BIN,
    }

    booster = lgb.train(params=params, train_set=dataset, num_boost_round=int(config["num_boost_round"]))
    del dataset, seq, y
    gc.collect()
    return booster


def predict_tree_window(booster, fit_start, fit_end, assess_start, assess_end):
    rules = learn_tree_category_rules(fit_start, fit_end)
    seq = TreeParquetSequence(assess_start, assess_end, rules)
    preds = []
    batch_size = 8192
    for start in range(0, len(seq), batch_size):
        stop = min(len(seq), start + batch_size)
        preds.append(booster.predict(seq[start:stop]))
    return np.concatenate(preds) if preds else np.array([], dtype=float)


def evaluate_tree_candidate(config, train_start, train_end, assess_start, assess_end):
    split = (train_start, train_end, assess_start, assess_end)
    summary_path, weekly_path = checkpoint_paths("tree", config, split)

    if summary_path.exists() and weekly_path.exists():
        with open(summary_path, encoding="utf-8") as f:
            return json.load(f), pd.read_csv(weekly_path)

    started = time.perf_counter()

    booster = fit_tree_booster(
        config,
        train_start,
        train_end,
    )

    pred = predict_tree_window(
        booster,
        train_start,
        train_end,
        assess_start,
        assess_end,
    )

    mask = train_meta[WEEK_COL].between(
        assess_start,
        assess_end,
    )

    y = train_meta.loc[
        mask,
        TARGET_COL,
    ].to_numpy(dtype=np.int8)

    weeks = train_meta.loc[
        mask,
        WEEK_COL,
    ].to_numpy(dtype=np.int16)

    if len(pred) != len(y):
        raise RuntimeError(
            f"Tree prediction length mismatch: pred={len(pred)} y={len(y)}"
        )

    overall_auc = auc_from_pred(y, pred)
    overall_gini = (
        float(2.0 * overall_auc - 1.0)
        if np.isfinite(overall_auc)
        else np.nan
    )

    calibration = calibration_metrics(y, pred)
    calibration_pass = calibration_guardrail(calibration)

    weekly, weekly_summary = weekly_metrics(
        y,
        pred,
        weeks,
    )

    summary = {
        **config,
        "branch": "tree",
        "train_start": int(train_start),
        "train_end": int(train_end),
        "assess_start": int(assess_start),
        "assess_end": int(assess_end),
        "auc": float(overall_auc),
        "gini": float(overall_gini),
        **calibration,
        "calibration_guardrail_pass": bool(calibration_pass),
        **weekly_summary,
        "fit_seconds": float(time.perf_counter() - started),
        "feature_count": len(TREE_FEATURES),
    }

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2,
        )

    weekly.to_csv(
        weekly_path,
        index=False,
    )

    return summary, weekly


## 12. Tree Stage 1 — Candidate Diagnostics

几个典型方向可以这样理解：

| Candidate 类型 | 实验问题 |
|---|---|
| `tree_baseline` | 当前参考配置表现怎样？ |
| leaves 更少 | 更简单的树是否更稳定？ |
| `min_data_in_leaf` 更大 | 禁止形成很小的叶子是否改善泛化？ |
| leaves 更多 | 更复杂的树是否真的有收益？ |
| `feature_fraction` 更低 | 每棵树随机少看一些特征是否降低过拟合？ |
| `bagging_fraction` 更低 | 随机少看一部分样本是否改善稳定性？ |
| L1/L2 更强 | 更强正则化是否有帮助？ |
| 小 learning rate + 更多 rounds | 更慢、更细的 boosting 是否更好？ |

Stage 1 仍然只使用：

```text
0–55 → 56–63
```

来快速淘汰明显没有竞争力的方向。


In [11]:
#@title 第 11 步：Tree Stage 1 — recent-window coarse screen
# ============================================================
# 11. Tree coarse screen
# ============================================================

tree_coarse = pd.DataFrame()

if RUN_TREE_COARSE and TREE_READY:
    rows = []

    for cfg in TREE_CANDIDATES:
        print("Tree Stage 1:", cfg["candidate"])

        summary, _ = evaluate_tree_candidate(
            cfg,
            STAGE1_TRAIN_START,
            STAGE1_TRAIN_END,
            STAGE1_ASSESS_START,
            STAGE1_ASSESS_END,
        )

        rows.append(summary)

        print(
            f"  AUC={summary['auc']:.6f} | Gini={summary['gini']:.6f} "
            f"| cal_slope={summary['calibration_slope']:.3f} "
            f"| cal_intercept={summary['calibration_intercept']:.3f} "
            f"| O/E={summary['oe_ratio']:.3f} "
            f"| cal_guardrail={summary['calibration_guardrail_pass']} "
            f"| {summary['fit_seconds']:.1f}s"
        )

    # Stage 1 仍以 AUC 为主。
    tree_coarse = pd.DataFrame(rows).sort_values(
        ["auc", "weekly_auc_sd"],
        ascending=[False, True],
    ).reset_index(drop=True)

    tree_coarse.to_csv(
        NB12_OUTPUT_DIR / "tree_coarse_results.csv",
        index=False,
    )

    display(tree_coarse)

elif (NB12_OUTPUT_DIR / "tree_coarse_results.csv").exists():
    tree_coarse = pd.read_csv(
        NB12_OUTPUT_DIR / "tree_coarse_results.csv"
    )
    display(tree_coarse)

else:
    print(
        "Tree Stage 1 skipped. 如果 TREE_READY=False，"
        "请先回 NB10 freeze tree__relevance__n500。"
    )


Tree Stage 1: tree_baseline
[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
  AUC=0.839347 | Gini=0.678695 | cal_slope=1.025 | cal_intercept=0.099 | O/E=1.035 | cal_guardrail=True | 186.7s
Tree Stage 1: tree_leaves15_leaf50
  AUC=0.836642 | Gini=0.673283 | cal_slope=1.055 | cal_intercept=0.182 | O/E=1.045 | cal_guardrail=True | 130.7s
Tree Stage 1: tree_leaves31_leaf50
  AUC=0.839592 | Gini=0.679184 | cal_slope=1.027 | cal_intercept=0.096 | O/E=1.029 | cal_guardrail=True | 135.4s
Tree Stage 1: tree_leaves31_leaf100
  AUC=0.839614 | Gini=0.679228 | cal_slope=1.027 | cal_intercept=0.093 | O/E=1.025 | cal_guardrail=True | 135.9s
Tree Stage 1: tree_leaves63_leaf50
  AUC=0.841383 | Gini=0.682766 

,learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,bagging_freq,lambda_l1,lambda_l2,num_boost_round,candidate,...,calibration_fit_success,calibration_guardrail_pass,mean_weekly_auc,weekly_auc_sd,weekly_auc_slope,mean_weekly_gini,weekly_gini_sd,weekly_gini_slope,fit_seconds,feature_count
0,0.05,63,50,0.8,0.8,1,0.0,1.0,300,tree_leaves63_leaf50,...,True,True,0.841447,0.008675,-0.000993,0.682895,0.017350,-0.001985,143.788391,500
1,0.05,63,100,0.8,0.8,1,0.0,1.0,300,tree_leaves63_leaf100,...,True,True,0.840970,0.009066,-0.000796,0.681939,0.018132,-0.001593,140.085364,500
2,0.03,31,20,0.8,0.8,1,0.0,1.0,500,tree_slow500,...,True,True,0.840773,0.008970,-0.000802,0.681546,0.017940,-0.001604,148.233790,500
3,0.05,31,20,0.6,0.8,1,0.0,1.0,300,tree_feature60,...,True,True,0.840366,0.009059,-0.000845,0.680732,0.018119,-0.001691,139.128079,500
4,0.05,31,20,0.8,0.8,1,0.0,5.0,300,tree_l2_5,...,True,True,0.839909,0.008864,-0.000579,0.679818,0.017729,-0.001159,139.614618,500
5,0.05,31,20,0.6,0.7,1,0.0,1.0,300,tree_subsample_both,...,True,True,0.839656,0.008434,-0.000726,0.679313,0.016868,-0.001453,141.905513,500
6,0.05,31,20,0.8,0.8,1,1.0,1.0,300,tree_l1_1_l2_1,...,True,True,0.839637,0.008078,-0.000683,0.679274,0.016156,-0.001366,137.762340,500
7,0.05,31,100,0.8,0.8,1,0.0,1.0,300,tree_leaves31_leaf100,...,True,True,0.839666,0.008836,-0.000635,0.679332,0.017673,-0.001270,135.925476,500
8,0.05,31,50,0.8,0.8,1,0.0,1.0,300,tree_leaves31_leaf50,...,True,True,0.839598,0.009166,-0.000867,0.679196,0.018331,-0.001735,135.414180,500
9,0.05,31,20,0.8,0.8,1,0.0,1.0,300,tree_baseline,...,True,True,0.839323,0.009099,-0.000780,0.678647,0.018197,-0.001561,186.716147,500


## 13. Tree F1–F4 — Temporal Robustness

树模型表达能力强，更容易出现：

```text
最近窗口很好
但早期 fold 不稳定
```

所以 Top Tree candidates 也必须重新做 F1–F4。

每一个 fold 都重新训练 LightGBM：

```text
新的 training window
→ 新的 categorical rules
→ 新的 booster
→ 预测未来 assessment window
```

最后比较：

```text
mean_fold_gini
worst_fold_gini
fold_gini_sd
weekly slope
```

如果两个 Tree 的平均 Gini 非常接近，会更偏向：

```text
worst fold 更高
+
波动更小
+
结构更简单
```

而不是机械选择最复杂的树。


In [12]:
#@title 第 12 步：Tree Top candidates → F1–F4 full temporal
# ============================================================
# 12. Tree full temporal validation
# ============================================================

tree_fold_results = pd.DataFrame()
tree_final_summary = pd.DataFrame()
tree_recommendations = {}

if RUN_TREE_FULL_TEMPORAL and TREE_READY and not tree_coarse.empty:
    promoted_ids = (
        tree_coarse
        .sort_values("auc", ascending=False)
        .head(TREE_PROMOTE_TOP_K)["candidate"]
        .tolist()
    )

    cfg_lookup = {
        cfg["candidate"]: cfg
        for cfg in TREE_CANDIDATES
    }

    print(
        "Tree promoted by Stage-1 AUC:",
        promoted_ids,
    )

    rows = []

    for fold_name, tr0, tr1, va0, va1 in TEMPORAL_FOLDS:
        for candidate in promoted_ids:
            print(
                "Tree full temporal:",
                candidate,
                fold_name,
            )

            summary, _ = evaluate_tree_candidate(
                cfg_lookup[candidate],
                tr0,
                tr1,
                va0,
                va1,
            )

            rows.append({
                "fold": fold_name,
                **summary,
            })

    tree_fold_results = pd.DataFrame(rows)

    tree_fold_results.to_csv(
        NB12_OUTPUT_DIR / "tree_full_temporal_folds.csv",
        index=False,
    )

    aggregate_rows = []

    for candidate, group in tree_fold_results.groupby(
        "candidate",
        sort=False,
    ):
        base = group.iloc[0]

        slope_dev = (
            group["calibration_slope"]
            - 1.0
        ).abs()

        intercept_abs = (
            group["calibration_intercept"]
            .abs()
        )

        oe_positive = (
            group["oe_ratio"]
            .where(group["oe_ratio"] > 0)
        )

        abs_log_oe = (
            np.log(oe_positive)
            .abs()
        )

        aggregate_rows.append({
            "candidate": candidate,
            "learning_rate": float(base["learning_rate"]),
            "num_leaves": int(base["num_leaves"]),
            "min_data_in_leaf": int(base["min_data_in_leaf"]),
            "feature_fraction": float(base["feature_fraction"]),
            "bagging_fraction": float(base["bagging_fraction"]),
            "lambda_l1": float(base["lambda_l1"]),
            "lambda_l2": float(base["lambda_l2"]),
            "num_boost_round": int(base["num_boost_round"]),

            # Primary metric family
            "mean_fold_auc": float(group["auc"].mean()),
            "worst_fold_auc": float(group["auc"].min()),
            "fold_auc_sd": float(group["auc"].std(ddof=0)),

            # Gini retained for continuity
            "mean_fold_gini": float(group["gini"].mean()),
            "worst_fold_gini": float(group["gini"].min()),
            "fold_gini_sd": float(group["gini"].std(ddof=0)),

            # Temporal ranking stability
            "mean_weekly_auc": float(group["mean_weekly_auc"].mean()),
            "mean_abs_weekly_auc_slope": float(
                group["weekly_auc_slope"].abs().mean()
            ),

            # Calibration diagnostics
            "mean_calibration_slope": float(
                group["calibration_slope"].mean()
            ),
            "calibration_slope_sd": float(
                group["calibration_slope"].std(ddof=0)
            ),
            "worst_calibration_slope_abs_dev": float(
                slope_dev.max()
            ),

            "mean_calibration_intercept": float(
                group["calibration_intercept"].mean()
            ),
            "calibration_intercept_sd": float(
                group["calibration_intercept"].std(ddof=0)
            ),
            "worst_calibration_intercept_abs": float(
                intercept_abs.max()
            ),

            "mean_oe_ratio": float(
                group["oe_ratio"].mean()
            ),
            "oe_ratio_sd": float(
                group["oe_ratio"].std(ddof=0)
            ),
            "worst_abs_log_oe": float(
                abs_log_oe.max()
            ),

            "calibration_guardrail_pass": bool(
                group["calibration_guardrail_pass"].all()
            ),
            "all_calibration_fits_successful": bool(
                group["calibration_fit_success"].all()
            ),

            "total_fit_seconds": float(
                group["fit_seconds"].sum()
            ),
            "feature_count": int(
                group["feature_count"].max()
            ),
        })

    tree_final_summary = (
        pd.DataFrame(aggregate_rows)
        .sort_values(
            [
                "mean_fold_auc",
                "worst_fold_auc",
                "fold_auc_sd",
            ],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    tree_final_summary.to_csv(
        NB12_OUTPUT_DIR / "tree_final_temporal_summary.csv",
        index=False,
    )

    tree_recommendations = build_dual_recommendations(
        tree_final_summary,
        "tree",
    )

    display(tree_final_summary)

    print(
        "Tree AUC_MAX:",
        tree_recommendations["AUC_MAX"],
    )

    print(
        "Tree ROBUST_1SE:",
        tree_recommendations["ROBUST_1SE"],
    )

    with open(
        NB12_OUTPUT_DIR / "tree_dual_recommendations.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            tree_recommendations,
            f,
            ensure_ascii=False,
            indent=2,
        )

else:
    print("Tree full temporal skipped.")


Tree promoted by Stage-1 AUC: ['tree_leaves63_leaf50', 'tree_leaves63_leaf100', 'tree_slow500', 'tree_feature60', 'tree_l2_5']
Tree full temporal: tree_leaves63_leaf50 F1
Tree full temporal: tree_leaves63_leaf100 F1
Tree full temporal: tree_slow500 F1
Tree full temporal: tree_feature60 F1
Tree full temporal: tree_l2_5 F1
Tree full temporal: tree_leaves63_leaf50 F2
Tree full temporal: tree_leaves63_leaf100 F2
Tree full temporal: tree_slow500 F2
Tree full temporal: tree_feature60 F2
Tree full temporal: tree_l2_5 F2
Tree full temporal: tree_leaves63_leaf50 F3
Tree full temporal: tree_leaves63_leaf100 F3
Tree full temporal: tree_slow500 F3
Tree full temporal: tree_feature60 F3
Tree full temporal: tree_l2_5 F3
Tree full temporal: tree_leaves63_leaf50 F4
Tree full temporal: tree_leaves63_leaf100 F4
Tree full temporal: tree_slow500 F4
Tree full temporal: tree_feature60 F4
Tree full temporal: tree_l2_5 F4


,candidate,learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,lambda_l1,lambda_l2,num_boost_round,mean_fold_auc,...,mean_calibration_intercept,calibration_intercept_sd,worst_calibration_intercept_abs,mean_oe_ratio,oe_ratio_sd,worst_abs_log_oe,calibration_guardrail_pass,all_calibration_fits_successful,total_fit_seconds,feature_count
0,tree_slow500,0.03,31,20,0.8,0.8,0.0,1.0,500,0.837791,...,0.088475,0.041538,0.135039,1.040231,0.030190,0.083995,True,True,503.510938,500
1,tree_leaves63_leaf50,0.05,63,50,0.8,0.8,0.0,1.0,300,0.837751,...,-0.006726,0.025725,0.043043,1.057241,0.032683,0.093369,True,True,588.044805,500
2,tree_feature60,0.05,31,20,0.6,0.8,0.0,1.0,300,0.837511,...,0.054171,0.033005,0.093037,1.032231,0.025818,0.068597,True,True,473.228547,500
3,tree_leaves63_leaf100,0.05,63,100,0.8,0.8,0.0,1.0,300,0.837493,...,-0.013165,0.043035,0.067891,1.051645,0.030006,0.092428,True,True,480.752070,500
4,tree_l2_5,0.05,31,20,0.8,0.8,0.0,5.0,300,0.837058,...,0.065308,0.039404,0.107014,1.048976,0.031155,0.091174,True,True,465.147953,500


Tree AUC_MAX: {'policy': 'AUC_MAX', 'candidate': 'tree_slow500', 'mean_fold_auc': 0.8377905948886761, 'worst_fold_auc': 0.8328454520849298, 'fold_auc_sd': 0.003728816886538342, 'mean_fold_gini': 0.6755811897773523, 'calibration_guardrail_pass': True, 'selection_rule': 'highest mean F1–F4 AUC -> highest worst-fold AUC -> lower fold AUC SD -> simpler tie-break', 'note': 'Competition-oriented policy. Calibration is reported but does not override the AUC winner.'}
Tree ROBUST_1SE: {'policy': 'ROBUST_1SE', 'candidate': 'tree_slow500', 'mean_fold_auc': 0.8377905948886761, 'worst_fold_auc': 0.8328454520849298, 'fold_auc_sd': 0.003728816886538342, 'mean_fold_gini': 0.6755811897773523, 'best_mean_fold_auc': 0.8377905948886761, 'best_fold_auc_sd': 0.003728816886538342, 'se_style_plateau_width': 0.001864408443269171, 'auc_plateau_threshold': 0.835926186445407, 'plateau_candidate_count': 5, 'calibration_guardrail_pass': True, 'calibration_status': 'PASS_FILTER_APPLIED', 'manual_review_required': F

## 13.1 候选结果判读

本册不采用“单个最高 Gini 自动胜出”。

### LR

顺序是：

```text
1. 是否全部 F1–F4 正常收敛
2. mean fold Gini
3. worst fold Gini
4. fold SD
5. weekly slope
6. 如果非常接近，偏向更强 regularization / 更简单方案
```

### Tree

顺序是：

```text
1. mean fold Gini
2. worst fold Gini
3. fold SD
4. weekly slope
5. 如果非常接近，偏向更简单树结构
```

自动 recommendation 使用 `best mean Gini - 0.01` 的近优平台，再在平台内选更稳健者。它是一个工程化的 robustness rule，不是统计置信区间。


## 14. AUC_MAX 与 ROBUST_1SE 两种选择策略

针对排序能力优先与时间稳健性优先两种目标，本实验预先定义独立的候选排序策略。

---

### Policy A：AUC_MAX

它回答：

> **如果我完全以竞赛 ranking metric 为优先，我应该选哪个配置？**

规则非常简单：

```text
LR:
先要求正常收敛

↓
mean F1–F4 AUC 最大

↓
如果完全相同：
worst-fold AUC 更高

↓
再相同：
fold AUC SD 更小

↓
最后：
更简单
```

Calibration slope / intercept / O/E 在这条 policy 中：

```text
只记录
不推翻 AUC winner
```

该策略作为 **competition-oriented benchmark**。

---

### Policy B：ROBUST_1SE

它回答：

> **如果 mean AUC 已经足够接近，我能不能选一个在未来时间上更稳健的配置？**

#### 第一步：建立 1-SE-style AUC plateau

```text
best mean F1–F4 AUC
↓
best candidate 的 fold AUC SD / sqrt(4)
↓
得到 SE-style width
↓
mean AUC >= best - width
→ 进入 plateau
```

这里仍然强调：

> temporal folds 的训练窗口有重叠，所以这是工程上的 **1-SE-style heuristic**，不是严格独立 CV standard error。

#### 第二步：进入 plateau 后，不再追几个千分点的 mean AUC

这是和 AUC_MAX 最大的区别。

一旦进入 plateau：

```text
0.790
0.788
0.786
```

不会继续机械地说：

```text
0.790 必然最好
```

而是认为：

> 这些候选已经达到 near-best discrimination 区域。

#### 第三步：Calibration 只做 Pass / Fail

三个 calibration 指标不加权：

```text
slope
intercept
O/E
```

只共同回答：

```text
这个概率模型有没有明显失真？
```

通过：

```text
PASS
```

明显越界：

```text
FAIL
```

如果 plateau 内存在 PASS candidate，FAIL candidate 不进入 robust winner 竞争。

#### 第四步：真正决定 Robust winner 的主要指标

在 calibration PASS 的 plateau candidate 中：

```text
① highest worst-fold AUC
→ 最差时间段也不要崩

② lower fold AUC SD
→ 不同时间段波动更小

③ simpler model
→ 性能与稳定性接近时不要无谓增加复杂度

④ mean AUC
→ 只保留成最后的极末端 tie-break
```

注意：

```text
weekly slope
calibration slope 数值差异
intercept 数值差异
O/E 数值差异
```

都继续保留做 diagnostic，

但**不继续堆进一个复杂加权总分**。

---

### 为什么这比多指标加权更容易解释？

我们没有定义：

```text
AUC 40%
Calibration 30%
Stability 20%
Complexity 10%
```

因为这些权重很难证明。

而是让每组指标承担不同角色：

```text
AUC
→ 资格区 / competition benchmark

Calibration
→ Pass / Fail 门槛

Worst-fold AUC
→ Robust policy 主指标

Fold SD
→ 稳定性第二指标

Complexity
→ tie-break
```

因此每一步都有清楚的业务和统计含义。

---

## 最终 NB12 会输出什么？

LR 最多两个配置：

```text
LR_AUC_MAX
LR_ROBUST_1SE
```

Tree 最多两个配置：

```text
TREE_AUC_MAX
TREE_ROBUST_1SE
```

它们可能：

```text
完全相同
```

也可能：

```text
不同
```

如果相同：

> 说明追求最高 development AUC 和追求 robust selection 得到了同一个配置。

如果不同：

> 两者可以在后续 Validation 阶段接受比较，但必须明确该时间窗口是否已在早期模型选择（例如历史基准的 early stopping）中使用。

两套策略共享模型训练结果，后续验证集比较应明确区分候选生成与最终模型选择。


## 15. Fast LR 搜索范围与计算预算

Fast LR 做的是：

```text
固定 solver / penalty
↓
只搜索 C
```

因此最终结论应该写成：

> **best LR regularization within the tested L2 + LBFGS search space**

而不要写成：

> “全局最优 Logistic Regression 超参数”。

这和 NB10 里“best within tested range”的表达原则是一致的。

这样做的理由是：

```text
百万级样本
+
456 维 Local FE design
+
SAGA 实际运行成本过高
```

而本项目更重要的是完成：

```text
temporal validation
calibration checks
AUC_MAX vs ROBUST_1SE
Tree comparison
future Validation / OOT
```

因此减少 optimizer 搜索是一个明确的计算预算取舍，这一取舍并不构成全局超参数最优性的证据。


In [13]:
#@title 第 13 步：汇总 NB12 双 Policy 最终候选
# ============================================================
# 13. Final NB12 dual-policy summary
# ============================================================

print("=" * 96)
print("NB12 FINAL SUMMARY — AUC_MAX vs ROBUST_1SE")
print("=" * 96)

if not lr_final_summary.empty:
    print("\nLR temporal summary")
    display(lr_final_summary)

    print("\nLR — Policy A: AUC_MAX")
    print(
        lr_recommendations.get(
            "AUC_MAX"
        )
    )

    print("\nLR — Policy B: ROBUST_1SE")
    print(
        lr_recommendations.get(
            "ROBUST_1SE"
        )
    )

else:
    print("\nLR: no final summary yet")


if not tree_final_summary.empty:
    print("\nTree temporal summary")
    display(tree_final_summary)

    print("\nTree — Policy A: AUC_MAX")
    print(
        tree_recommendations.get(
            "AUC_MAX"
        )
    )

    print("\nTree — Policy B: ROBUST_1SE")
    print(
        tree_recommendations.get(
            "ROBUST_1SE"
        )
    )

else:
    print("\nTree: no final summary yet")


handoff_preview = {
    "nb12_version": NB12_EVALUATOR_VERSION,
    "development_only": True,
    "development_weeks": [0, 63],
    "selection_policies": {
        "AUC_MAX": (
            "Competition-oriented: highest mean F1–F4 AUC."
        ),
        "ROBUST_1SE": (
            "1-SE-style AUC plateau -> calibration Pass/Fail -> "
            "highest worst-fold AUC -> lower fold AUC SD -> simpler model."
        ),
    },
    "strict_independent_cv_se_claimed": False,
    "calibration_guardrails": {
        "slope_max_abs_dev": CAL_SLOPE_MAX_ABS_DEV,
        "intercept_max_abs": CAL_INTERCEPT_MAX_ABS,
        "oe_max_multiplicative_deviation": OE_MAX_MULTIPLICATIVE_DEVIATION,
    },
    "validation_read": False,
    "oot_read": False,
    "lr_representation_package": (
        LR_HANDOFF.get("final_package")
        if LR_READY
        else None
    ),
    "tree_feature_count": (
        len(TREE_FEATURES)
        if TREE_READY
        else None
    ),
    "lr_recommendations": (
        lr_recommendations
        if not lr_final_summary.empty
        else {}
    ),
    "tree_recommendations": (
        tree_recommendations
        if not tree_final_summary.empty
        else {}
    ),
    "next_notebook": (
        "NB13 compare AUC_MAX vs ROBUST_1SE "
        "on reserved Validation, then LR vs Tree vs Ensemble"
    ),
}

with open(
    NB12_OUTPUT_DIR / "nb12_dual_policy_summary.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        handoff_preview,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "\nSaved:",
    NB12_OUTPUT_DIR
    / "nb12_dual_policy_summary.json",
)


NB12 FINAL SUMMARY — AUC_MAX vs ROBUST_1SE

LR temporal summary


,candidate,solver,penalty,C,mean_fold_auc,worst_fold_auc,fold_auc_sd,mean_fold_gini,worst_fold_gini,fold_gini_sd,...,worst_calibration_intercept_abs,mean_oe_ratio,oe_ratio_sd,worst_abs_log_oe,calibration_guardrail_pass,all_calibration_fits_successful,all_converged,max_n_iter,total_fit_seconds,design_columns
0,lr_l2_lbfgs_c0p03,lbfgs,l2,0.03,0.789451,0.759682,0.017438,0.578902,0.519365,0.034877,...,1.677512,0.859415,0.318594,0.937712,False,True,True,92,205.843081,456
1,lr_l2_lbfgs_c3,lbfgs,l2,3.00,0.787463,0.750888,0.021178,0.574927,0.501776,0.042356,...,1.922901,0.871960,0.393109,1.109859,False,True,True,109,256.924056,456
2,lr_l2_lbfgs_c1,lbfgs,l2,1.00,0.787447,0.750547,0.021441,0.574895,0.501095,0.042881,...,1.930754,0.873679,0.379915,1.112587,False,True,True,126,250.891404,456



LR — Policy A: AUC_MAX
{'policy': 'AUC_MAX', 'candidate': 'lr_l2_lbfgs_c0p03', 'mean_fold_auc': 0.7894512457507447, 'worst_fold_auc': 0.7596823275089049, 'fold_auc_sd': 0.017438252151611103, 'mean_fold_gini': 0.5789024915014892, 'calibration_guardrail_pass': False, 'selection_rule': 'highest mean F1–F4 AUC -> highest worst-fold AUC -> lower fold AUC SD -> simpler tie-break', 'note': 'Competition-oriented policy. Calibration is reported but does not override the AUC winner.'}

LR — Policy B: ROBUST_1SE
{'policy': 'ROBUST_1SE', 'candidate': 'lr_l2_lbfgs_c0p03', 'mean_fold_auc': 0.7894512457507447, 'worst_fold_auc': 0.7596823275089049, 'fold_auc_sd': 0.017438252151611103, 'mean_fold_gini': 0.5789024915014892, 'best_mean_fold_auc': 0.7894512457507447, 'best_fold_auc_sd': 0.017438252151611103, 'se_style_plateau_width': 0.008719126075805552, 'auc_plateau_threshold': 0.780732119674939, 'plateau_candidate_count': 3, 'calibration_guardrail_pass': False, 'calibration_status': 'NO_PASSING_CANDID

,candidate,learning_rate,num_leaves,min_data_in_leaf,feature_fraction,bagging_fraction,lambda_l1,lambda_l2,num_boost_round,mean_fold_auc,...,mean_calibration_intercept,calibration_intercept_sd,worst_calibration_intercept_abs,mean_oe_ratio,oe_ratio_sd,worst_abs_log_oe,calibration_guardrail_pass,all_calibration_fits_successful,total_fit_seconds,feature_count
0,tree_slow500,0.03,31,20,0.8,0.8,0.0,1.0,500,0.837791,...,0.088475,0.041538,0.135039,1.040231,0.030190,0.083995,True,True,503.510938,500
1,tree_leaves63_leaf50,0.05,63,50,0.8,0.8,0.0,1.0,300,0.837751,...,-0.006726,0.025725,0.043043,1.057241,0.032683,0.093369,True,True,588.044805,500
2,tree_feature60,0.05,31,20,0.6,0.8,0.0,1.0,300,0.837511,...,0.054171,0.033005,0.093037,1.032231,0.025818,0.068597,True,True,473.228547,500
3,tree_leaves63_leaf100,0.05,63,100,0.8,0.8,0.0,1.0,300,0.837493,...,-0.013165,0.043035,0.067891,1.051645,0.030006,0.092428,True,True,480.752070,500
4,tree_l2_5,0.05,31,20,0.8,0.8,0.0,5.0,300,0.837058,...,0.065308,0.039404,0.107014,1.048976,0.031155,0.091174,True,True,465.147953,500



Tree — Policy A: AUC_MAX
{'policy': 'AUC_MAX', 'candidate': 'tree_slow500', 'mean_fold_auc': 0.8377905948886761, 'worst_fold_auc': 0.8328454520849298, 'fold_auc_sd': 0.003728816886538342, 'mean_fold_gini': 0.6755811897773523, 'calibration_guardrail_pass': True, 'selection_rule': 'highest mean F1–F4 AUC -> highest worst-fold AUC -> lower fold AUC SD -> simpler tie-break', 'note': 'Competition-oriented policy. Calibration is reported but does not override the AUC winner.'}

Tree — Policy B: ROBUST_1SE
{'policy': 'ROBUST_1SE', 'candidate': 'tree_slow500', 'mean_fold_auc': 0.8377905948886761, 'worst_fold_auc': 0.8328454520849298, 'fold_auc_sd': 0.003728816886538342, 'mean_fold_gini': 0.6755811897773523, 'best_mean_fold_auc': 0.8377905948886761, 'best_fold_auc_sd': 0.003728816886538342, 'se_style_plateau_width': 0.001864408443269171, 'auc_plateau_threshold': 0.835926186445407, 'plateau_candidate_count': 5, 'calibration_guardrail_pass': True, 'calibration_status': 'PASS_FILTER_APPLIED', 'ma

## 16. 候选配置冻结与下游使用

Notebook 10 固定 Tree feature subset，Notebook 11 固定 LR representation；Notebook 12 根据 development 内部的 temporal folds 对模型配置进行评估。

本次 LR 搜索范围为 `solver=lbfgs`、`penalty=l2` 及 `C ∈ {0.03, 0.1, 0.3, 1, 3}`；Tree 候选则比较复杂度、采样比例、正则化强度与 boosting rounds。两种策略分别基于 `AUC_MAX` 与 `ROBUST_1SE` 生成候选建议。

**实际执行状态：** Notebook 末尾保存的输出显示手动冻结开关未开启。因此不能仅依据本册的 Markdown 声称已导出最终冻结配置。最终采用的参数以及在 Week 64–77 的表现，需参照后续 Notebook 13–14 的实际结果。

本阶段不包含 OOT Week 78–91 的最终评估。


In [14]:
#@title 第 14 步：Freeze 两条 Policy 给 Notebook 13
# ============================================================
# 14. Freeze dual policy candidates for NB13
# ============================================================

if FREEZE_FINAL_MODELS:
    if lr_final_summary.empty and tree_final_summary.empty:
        raise RuntimeError(
            "NB12 还没有完成任何 F1–F4 summary，不能 Freeze。"
        )

    lr_lookup = {
        cfg["candidate"]: cfg
        for cfg in LR_CANDIDATES
    }

    tree_lookup = {
        cfg["candidate"]: cfg
        for cfg in TREE_CANDIDATES
    }

    frozen = {
        "lr": {},
        "tree": {},
    }

    # --------------------------------------------------------
    # LR: freeze AUC_MAX and ROBUST_1SE candidates
    # --------------------------------------------------------
    if not lr_final_summary.empty:
        for policy_name in [
            "AUC_MAX",
            "ROBUST_1SE",
        ]:
            rec = (
                lr_recommendations
                .get(policy_name)
            )

            if rec is None:
                continue

            candidate = rec["candidate"]

            if candidate not in lr_lookup:
                raise KeyError(
                    f"Unknown LR candidate: {candidate}"
                )

            row = lr_final_summary[
                lr_final_summary["candidate"]
                == candidate
            ]

            if row.empty:
                raise RuntimeError(
                    f"LR {policy_name} candidate "
                    "没有完成 F1–F4。"
                )

            row = row.iloc[0]

            if not bool(row["all_converged"]):
                raise RuntimeError(
                    f"LR {policy_name} candidate "
                    "并非全部 folds 收敛。"
                )

            frozen["lr"][policy_name] = {
                "candidate": candidate,
                "model_config": (
                    lr_lookup[candidate]
                ),
                "selection_record": rec,
            }

    # --------------------------------------------------------
    # Tree: freeze AUC_MAX and ROBUST_1SE candidates
    # --------------------------------------------------------
    if not tree_final_summary.empty:
        for policy_name in [
            "AUC_MAX",
            "ROBUST_1SE",
        ]:
            rec = (
                tree_recommendations
                .get(policy_name)
            )

            if rec is None:
                continue

            candidate = rec["candidate"]

            if candidate not in tree_lookup:
                raise KeyError(
                    f"Unknown Tree candidate: {candidate}"
                )

            frozen["tree"][policy_name] = {
                "candidate": candidate,
                "model_config": (
                    tree_lookup[candidate]
                ),
                "selection_record": rec,
            }

    nb13_handoff = {
        "selection_design": (
            "Dual policy comparison defined before Validation read"
        ),
        "policies": {
            "AUC_MAX": (
                "Competition-oriented winner by mean F1–F4 AUC."
            ),
            "ROBUST_1SE": (
                "1-SE-style plateau; calibration Pass/Fail; "
                "worst-fold AUC; fold SD; simplicity."
            ),
        },
        "lr_representation_from_nb11": (
            LR_HANDOFF.get("final_package")
            if LR_READY
            else None
        ),
        "tree_feature_file_from_nb10": (
            "selected_features_tree.json"
            if TREE_READY
            else None
        ),
        "frozen_candidates": frozen,
        "development_search_complete": True,
        "validation_read": False,
        "oot_read": False,
        "next_notebook": (
            "NB13: compare AUC_MAX vs ROBUST_1SE "
            "on Week 64–77, then compare LR / Tree / Ensemble."
        ),
    }

    with open(
        NB12_OUTPUT_DIR / "nb13_dual_policy_handoff.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            nb13_handoff,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("Dual-policy freeze complete.")
    print(json.dumps(
        frozen,
        ensure_ascii=False,
        indent=2,
    ))

    print(
        "\nNB13 handoff:",
        NB12_OUTPUT_DIR
        / "nb13_dual_policy_handoff.json",
    )

else:
    print("Manual freeze disabled.")
    print(
        "先完成 LR / Tree F1–F4，"
        "然后设置 FREEZE_FINAL_MODELS=True。"
    )
    print(
        "这一步会同时 Freeze AUC_MAX 与 ROBUST_1SE，"
        "不是在 NB12 强行决定唯一最终 winner。"
    )


Manual freeze disabled.
先完成 LR / Tree F1–F4，然后设置 FREEZE_FINAL_MODELS=True。
这一步会同时 Freeze AUC_MAX 与 ROBUST_1SE，不是在 NB12 强行决定唯一最终 winner。


## 17. 实验方法摘要

```text
Upstream Feature / Representation Freeze
                ↓
Knowledge-guided Hyperparameter Candidates
                ↓
Development-only Recent-window Screening
                ↓
F1–F4 Temporal Evaluation
                ↓
        Shared Candidate Results
                ↓
        ┌───────┴────────┐
        ▼                ▼
     AUC_MAX        ROBUST_1SE
  Mean AUC first   Near-best AUC plateau
                   + Calibration guardrail
                   + Worst-fold AUC / Variability
        └───────┬────────┘
                ▼
     Downstream Validation Comparison
                ▼
     Final Model Selection / OOT Assessment
```

`ROBUST_1SE` 中的 1-SE-style plateau 是一种工程近似方法。Expanding-window folds 的训练样本重叠，因此其 width **不应解释为独立交叉验证的严格标准误差**。

完整模型最终选择的具体阶段在后续 Notebook 中记录；本册不使用 Week 64–77 或 Week 78–91。


## 18. 两种 Policy 的可复核定义

| 规则 | 优先目标 | 排序逻辑 | 用途 |
|---|---|---|---|
| `AUC_MAX` | 开发期排序能力 | 收敛前提下最大化 mean F1–F4 AUC；随后比较 worst-fold AUC 与方差 | Competition-oriented benchmark |
| `ROBUST_1SE` | 接近最佳 AUC 时的稳健性 | 限定于 1-SE-style plateau；校准诊断作为 Pass/Fail；优先 worst-fold AUC、更低波动及更简单的配置 | Robustness-oriented challenger |

两种规则使用相同的候选训练结果，不需要为了比较 policy 再次拟合模型。模型最终优劣须根据独立的下游评估和既定选择协议判断，而非仅凭 development 内部得分。
